# A

In [ ]:
# ==========================
# Block A: GLM Pipeline — Environment Setup & Path Constants
# Version: v1.0
# Purpose: Install required packages, mount Drive, define all GLM pipeline
#          path constants, fix RNG seeds, and configure logging.
# ==========================
#
# What this block does:
# - Feature 1 (Dependency Check ✅): Checks whether transformers>=4.51.0,
#   bitsandbytes>=0.46.0, accelerate, peft, and scikit-learn are installed
#   at the expected versions. If any are missing or mismatched, installs them
#   in the correct order and performs a hard os.kill(os.getpid(), 9) to force
#   Colab to reload the Python runtime. This is the same pattern proven in
#   Block 1 of the training pipeline: pip skips already-correct packages, so
#   subsequent runs take ~10 seconds instead of 5 minutes. The kill-and-restart
#   strategy is the only reliable way to load freshly installed C extensions
#   (bitsandbytes CUDA kernels, tokenizers Rust backend) inside Colab.
# - Feature 2 (Drive Mount ✅): Mounts Google Drive at /content/drive using
#   drive.mount() with no force_remount so already-mounted sessions skip it.
#   All pipeline paths live under /content/drive/MyDrive/Twitter/Finetune/
#   exactly matching the training pipeline's BASE_DIR convention.
# - Feature 3 (Path Constants ✅): Defines INPUT_TRAIN, INPUT_FN, OUTPUT,
#   CHECKPOINT, and REVIEW_PATH as module-level constants. OUTPUT is the new
#   glm_train_v3_4k.csv. CHECKPOINT is the atomic resume file written every
#   100 tweets. REVIEW_PATH holds parse failures and low-confidence rows for
#   manual inspection after the run. All paths are printed at startup for
#   quick visual verification before the long inference run begins.
# - Feature 4 (Reproducibility ✅): Seeds Python random, NumPy, and PyTorch
#   to 42 — same seed used in the training pipeline for consistency across
#   all pipeline stages. The GLM pipeline uses a deterministic shuffle in
#   Block B and this seed must be set before that call.
# - Feature 5 (Logging ✅): Creates the dei_glm logger with both a
#   FileHandler writing to training.log (same log file as the training
#   pipeline, appended not overwritten) and a StreamHandler to stdout. Uses
#   the same %(asctime)s | %(levelname)s | %(message)s format so log
#   entries from the GLM pipeline are distinguishable by timestamp from
#   training entries but live in the same file.
# - Feature 6 (CUDA Verification ✅): Asserts CUDA is available and logs
#   GPU name + VRAM. The Qwen3-14B 4-bit model requires ~8–9 GB VRAM,
#   leaving ~5–6 GB for activations and KV cache. Raises RuntimeError if
#   GPU is not available rather than silently proceeding to a CPU run that
#   would take days.
#
# How it works:
# - Step 1: Define _pip() and _check_version() helpers (identical to Block 1).
# - Step 2: Check three sentinel package versions. If any mismatch → install
#   all packages → print restart banner → os.kill.
# - Step 3: Import all standard libs (logging, os, random, numpy, torch).
# - Step 4: Mount Drive.
# - Step 5: Define and print all path constants. Create directories.
# - Step 6: Configure logger with FileHandler + StreamHandler.
# - Step 7: Seed all RNGs.
# - Step 8: Verify CUDA.
# - Step 9: HuggingFace login from Colab Secrets.
#
# Solutions attempted that did not work:
# - Failed attempt 1: Using !pip install inline without _check_version guard.
#   On every subsequent cell run the install step consumed 5 minutes even
#   when packages were already correct. The _check_version guard cuts this to
#   ~10 seconds on warm runs.
#
# Solutions implemented:
# - Solution 1: _check_version gate + os.kill restart pattern from Block 1.
#   The kill is intentional — do NOT remove it. Run the cell twice if you
#   see the "INTENTIONAL STOP" banner.

print("====++++====")
print("Block A Output")
print("====++++====")
print()

import subprocess, sys, os

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "--quiet", "--no-warn-script-location", *args])

def _check_version(pkg: str, required: str) -> bool:
    try:
        import importlib.metadata
        return importlib.metadata.version(pkg) == required
    except Exception:
        return False

_needs_install = (
    not _check_version("transformers", "4.51.0") or
    not _check_version("peft",         "0.15.0") or
    not _check_version("trl",          "0.13.0")
)

if _needs_install:
    print("Installing / upgrading packages — this takes ~5 min on first run...")
    _pip("transformers==4.51.0")
    _pip("peft==0.15.0")
    _pip("trl==0.13.0")
    _pip("bitsandbytes>=0.46.0")
    _pip("accelerate", "--upgrade")
    _pip("scikit-learn==1.5.2")
    _pip("pandas")
    print("\n" + "=" * 60)
    print("🛑 INTENTIONAL STOP 🛑")
    print("Packages installed. Click Play ONE MORE TIME to reload runtime.")
    print("=" * 60 + "\n")
    os.kill(os.getpid(), 9)
else:
    print("✅ All packages at correct versions — skipping install.")

import logging, random
import numpy as np
import torch
from google.colab import drive, userdata
import huggingface_hub

# ── Mount Drive ────────────────────────────────────────────────────────────────
drive.mount("/content/drive")

# ── Path Constants ─────────────────────────────────────────────────────────────
BASE_DIR    = "/content/drive/MyDrive/Twitter/Finetune"
DATA_DIR    = os.path.join(BASE_DIR, "data")
LOG_DIR     = os.path.join(BASE_DIR, "logs")
LOG_FILE    = os.path.join(LOG_DIR, "training.log")

INPUT_TRAIN = os.path.join(DATA_DIR, "processed", "merged_4k_v2_train.csv")
INPUT_FN    = os.path.join(DATA_DIR, "block9_candidates", "block9_v2_9M_candidates.csv")
OUTPUT      = os.path.join(DATA_DIR, "processed", "glm_train_v3_4k.csv")
CHECKPOINT  = os.path.join(DATA_DIR, "judge", "glm_train_v3_4k_checkpoint.csv")
REVIEW_PATH = os.path.join(DATA_DIR, "processed", "glm_train_v3_4k_review.csv")
os.makedirs(os.path.dirname(CHECKPOINT), exist_ok=True)
for _d in [os.path.join(DATA_DIR, "processed"), LOG_DIR]:
    os.makedirs(_d, exist_ok=True)

# ── Logging ────────────────────────────────────────────────────────────────────
_log_fmt = "%(asctime)s | %(levelname)s | %(message)s"
logger = logging.getLogger("dei_glm")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_fh = logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8")
_fh.setFormatter(logging.Formatter(_log_fmt))
logger.addHandler(_fh)
_sh = logging.StreamHandler(sys.stdout)
_sh.setFormatter(logging.Formatter(_log_fmt))
logger.addHandler(_sh)
logger.propagate = False

# ── Seeds ──────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
logger.info(f"RNG seeds fixed: {SEED}")

# ── CUDA Check ────────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available. GLM pipeline requires a GPU runtime.")
_gpu  = torch.cuda.get_device_name(0)
_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
logger.info(f"GPU: {_gpu} | VRAM: {_vram:.1f} GB")

# ── HuggingFace Auth ──────────────────────────────────────────────────────────
try:
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    raise RuntimeError("HF_TOKEN not found in Colab Secrets. Enable it in the Key icon sidebar.")
huggingface_hub.login(token=_hf_token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = _hf_token

logger.info(f"INPUT_TRAIN : {INPUT_TRAIN}")
logger.info(f"INPUT_FN    : {INPUT_FN}")
logger.info(f"OUTPUT      : {OUTPUT}")
logger.info(f"CHECKPOINT  : {CHECKPOINT}")
logger.info("✅ Block A complete.")

====++++====
Block A Output
====++++====

✅ All packages at correct versions — skipping install.
Mounted at /content/drive
2026-04-19 11:03:47,930 | INFO | RNG seeds fixed: 42
2026-04-19 11:03:48,648 | INFO | GPU: Tesla T4 | VRAM: 14.6 GB
2026-04-19 11:03:49,902 | INFO | INPUT_TRAIN : /content/drive/MyDrive/Twitter/Finetune/data/processed/merged_4k_v2_train.csv
2026-04-19 11:03:49,903 | INFO | INPUT_FN    : /content/drive/MyDrive/Twitter/Finetune/data/block9_candidates/block9_v2_9M_candidates.csv
2026-04-19 11:03:49,905 | INFO | OUTPUT      : /content/drive/MyDrive/Twitter/Finetune/data/processed/glm_train_v3_4k.csv
2026-04-19 11:03:49,906 | INFO | CHECKPOINT  : /content/drive/MyDrive/Twitter/Finetune/data/judge/glm_train_v3_4k_checkpoint.csv
2026-04-19 11:03:49,908 | INFO | ✅ Block A complete.


In [ ]:
# ==========================
# Block B v2.0: Load & Merge ALL Data — No sampling cap
# ==========================
print("====++++====\nBlock B v2.0 Output\n====++++====\n")

import os, logging
import pandas as pd
import numpy as np

logging.raiseExceptions = False
logger = logging.getLogger("dei_glm")

BASE_DIR      = "/content/drive/MyDrive/Twitter/Finetune"
INPUT_TRAIN   = os.path.join(BASE_DIR, "data/processed/merged_4k_v2_train.csv")
INPUT_FN      = os.path.join(BASE_DIR, "data/block9_candidates/block9_v2_9M_candidates.csv")
DISAGREE_PATH = os.path.join(BASE_DIR, "data/processed/Model_Disagreement_Analysis.csv")
SEED          = 42

# ── Load train ────────────────────────────────────────────────────────────────
logger.info("Loading train file...")
train_raw = pd.read_csv(INPUT_TRAIN, encoding_errors="replace")
train_df  = train_raw.rename(columns={"text": "Tweet", "label": "old_label"}).copy()
train_df["source"]   = "train"
train_df["category"] = "unknown"
train_df = train_df[["Tweet", "old_label", "source", "category"]]
train_df["old_label"] = train_df["old_label"].astype(int)
logger.info(f"Train: {len(train_df)} rows | label dist: {train_df['old_label'].value_counts().to_dict()}")

# ── Load FN candidates ─────────────────────────────────────────────────────────
logger.info("Loading FN candidates...")
fn_raw = pd.read_csv(INPUT_FN, encoding_errors="replace")
fn_df  = fn_raw.copy()
if "Tweet" not in fn_df.columns:
    fn_df = fn_df.rename(columns={"tweet": "Tweet"})
fn_df["old_label"] = 0
fn_df["source"]    = "fn"
if "category" not in fn_df.columns:
    fn_df["category"] = "unknown"
fn_df = fn_df[["Tweet", "old_label", "source", "category"]]
logger.info(f"FN: {len(fn_df)} rows")

# ── Load disagreement file ─────────────────────────────────────────────────────
disagree_df = pd.DataFrame()
if os.path.isfile(DISAGREE_PATH):
    logger.info("Loading disagreement file...")
    dis_raw  = pd.read_csv(DISAGREE_PATH, encoding_errors="replace")
    tweet_col = "Tweet_Text" if "Tweet_Text" in dis_raw.columns else "Tweet"
    if tweet_col in dis_raw.columns:
        dis_raw = dis_raw.rename(columns={tweet_col: "Tweet"})
        label_cols = [c for c in ["L_4B", "L_9B", "L_Gemini"] if c in dis_raw.columns]
        dis_raw["old_label"] = (
            dis_raw[label_cols].sum(axis=1) >= (len(label_cols) / 2)
        ).astype(int) if label_cols else 0
        dis_raw["source"]   = "disagree"
        dis_raw["category"] = "unknown"
        disagree_df = dis_raw[["Tweet", "old_label", "source", "category"]].copy()
        logger.info(f"Disagree: {len(disagree_df)} rows")
else:
    logger.warning(f"Disagreement file not found — skipping.")

# ── Merge all sources ──────────────────────────────────────────────────────────
frames = [train_df, fn_df]
if not disagree_df.empty:
    frames.append(disagree_df)

merged = pd.concat(frames, ignore_index=True)
logger.info(f"After concat: {len(merged)} rows | sources: {merged['source'].value_counts().to_dict()}")

# ── Deduplicate on Tweet text — keep first occurrence ─────────────────────────
before = len(merged)
# Normalize whitespace before dedup to catch near-duplicates
merged["_tweet_norm"] = merged["Tweet"].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)
merged = merged.drop_duplicates(subset=["_tweet_norm"], keep="first").drop(columns=["_tweet_norm"])
merged = merged.reset_index(drop=True)
logger.info(f"Dedup: dropped {before - len(merged)} duplicates | Remaining: {len(merged)}")

# ── Shuffle ────────────────────────────────────────────────────────────────────
# NO sampling — use everything
glm_df = merged.sample(frac=1, random_state=SEED).reset_index(drop=True)
logger.info(f"Final dataset: {len(glm_df)} rows (ALL data, no cap) | seed={SEED}")
logger.info(f"By source: {glm_df['source'].value_counts().to_dict()}")
logger.info(f"old_label dist: {glm_df['old_label'].value_counts().to_dict()}")
logger.info("✅ Block B v2.0 complete.")
print(glm_df.head(10).to_string())
print(f"\nSource breakdown:\n{glm_df['source'].value_counts()}")
print(f"\nTotal rows to classify: {len(glm_df):,}")

====++++====
Block B v2.0 Output
====++++====

2026-04-19 11:03:50,192 | INFO | Loading train file...
2026-04-19 11:03:50,739 | INFO | Train: 3884 rows | label dist: {1: 1988, 0: 1896}
2026-04-19 11:03:50,741 | INFO | Loading FN candidates...
2026-04-19 11:03:51,197 | INFO | FN: 1200 rows
2026-04-19 11:03:51,200 | INFO | Loading disagreement file...
2026-04-19 11:03:51,865 | INFO | Disagree: 400 rows
2026-04-19 11:03:51,870 | INFO | After concat: 5484 rows | sources: {'train': 3884, 'fn': 1200, 'disagree': 400}
2026-04-19 11:03:51,982 | INFO | Dedup: dropped 389 duplicates | Remaining: 5095
2026-04-19 11:03:51,986 | INFO | Final dataset: 5095 rows (ALL data, no cap) | seed=42
2026-04-19 11:03:51,990 | INFO | By source: {'train': 3883, 'fn': 812, 'disagree': 400}
2026-04-19 11:03:51,991 | INFO | old_label dist: {0: 3008, 1: 2087}
2026-04-19 11:03:51,993 | INFO | ✅ Block B v2.0 complete.
                                                                                                     

In [ ]:
# ==========================
# Block C-A: Load Judge Model — Tokenizer + Quantization Config + Model Load
# Version: v5.0
# Purpose: Load Qwen3-14B in 4-bit NF4.
#   - HF download (28 GB BF16) → Colab local disk (/root/.cache, ~78 GB available)
#   - Quantized result (~7-8 GB) → saved to Drive by Block C-B ONLY
#   - Drive is NEVER used for the raw BF16 download (fixes Drive-space exhaustion)
# ==========================
#
# KEY CHANGE vs v4.0:
#   v4.0 BUG: Lines 79-83 overrode LOCAL_CACHE with HF_CACHE_DIR (a Drive path),
#   causing the 28 GB BF16 download to land on Drive and fill it.
#   v5.0 FIX: HF env vars point ONLY to local Colab disk. Drive path is used
#   exclusively for SAVED_MODEL_PATH (the 7-8 GB quantized output written by C-B).
#
# Cache locations:
#   HF BF16 download  → /root/.cache/huggingface   (Colab local, ephemeral, ~78 GB)
#   Quantized weights → Drive/judge/model_weights/qwen3_14b  (permanent, ~7-8 GB)
#
# VRAM budget (T4, 14.6 GB total):
#   Model weights (NF4 4-bit):  ~7.8 GB alloc / ~9.3 GB reserved
#   KV cache for CoT inference: ~3-5 GB headroom
#   With batch_size=12 + enable_thinking=True + 1024 new tok/tweet → tight but OK
#   If VRAM reserved > 85%: E-C's dynamic batch sizer will reduce batch_size automatically
#
# Qwen3-27B on T4: NOT FEASIBLE. 27B NF4 4-bit = ~14-15 GB weights alone,
#   leaving <1 GB for KV cache. Requires A100 (40 GB) or L4 (24 GB).
#
# ==========================
print("====++++====")
print("Block C-A v5.0 Output")
print("====++++====")
print()

import os, logging, sys
import torch

logging.raiseExceptions = False
logger = logging.getLogger("dei_glm")

BASE_DIR         = "/content/drive/MyDrive/Twitter/Finetune"
SAVED_MODEL_PATH = os.path.join(BASE_DIR, "judge", "model_weights", "qwen3_14b")

# ── LOCAL Colab disk for ALL HF downloads (ephemeral, ~78 GB available) ──────
# CRITICAL: Set these BEFORE any HF/transformers import.
# These point to LOCAL disk, NOT Drive.
# The 28 GB BF16 download lives here and is lost on Colab disconnect — that's fine.
# Only the 7-8 GB quantized result (saved by Block C-B) needs to survive on Drive.
LOCAL_HF_CACHE = "/root/.cache/huggingface"
os.environ["HF_HOME"]               = LOCAL_HF_CACHE
os.environ["TRANSFORMERS_CACHE"]    = LOCAL_HF_CACHE
os.environ["HF_DATASETS_CACHE"]     = LOCAL_HF_CACHE
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(LOCAL_HF_CACHE, "hub")

# NOTE: We do NOT set HF env vars to any Drive path.
# Drive is only used for SAVED_MODEL_PATH (quantized weights written by C-B).
os.makedirs(SAVED_MODEL_PATH, exist_ok=True)

logger.info(f"HF BF16 cache (local Colab disk, ephemeral) → {LOCAL_HF_CACHE}")
logger.info(f"Quantized model path (Drive, permanent)     → {SAVED_MODEL_PATH}")

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

JUDGE_PRIMARY  = "Qwen/Qwen3-14B"
JUDGE_FALLBACK = "Qwen/Qwen2.5-14B-Instruct"

# ── System prompt ──────────────────────────────────────────────────────────────
JUDGE_SYSTEM_PROMPT = """You are an expert DEI (Diversity, Equity, and Inclusion) content classifier for university social media.

DEI CONTENT (label=1):
Content substantively engaging with diversity, equity, or inclusion as a social concept, policy, practice, or advocacy:
Racial/ethnic equity, anti-racism, BIPOC representation, civil rights
LGBTQ+ rights, identity affirmation, Pride, queer inclusion
Disability rights, accessibility, neurodiversity advocacy, ADA accommodations
Gender equity, feminism, women's rights, Title IX
Institutional DEI programs, diversity offices, inclusive pedagogy, affirmative action
Social justice: criminal justice reform, immigration/refugee rights, human rights
First-generation student access, low-income support, socioeconomic mobility

NOT DEI (label=0):
Administrative announcements: grading policies, financial aid disbursements, deadlines, registration
Generic supportive language WITHOUT DEI substance: "we welcome all students"
Biodiversity (ecological): "diversity of plant species" is NOT social DEI
"Opportunity" alone is NOT DEI — "research opportunity" or "job opportunity" are not DEI
General campus events, sports, alumni without explicit DEI focus
Academic research unrelated to social DEI
HBCU mentions without DEI substance: "FAMU homecoming game" = campus_life NOT DEI

CATEGORIES:
lgbtq | race_ethnicity | disability | gender | general_dei | social_justice | first_gen | admin | academic | campus_life | other

CONFIDENCE CALIBRATION:
Confidence measures how certain you are about the label you assigned — applies equally to label=0 and label=1.

- 95–100: You are certain. The tweet is unambiguously one label with no reasonable alternative.
          Examples of certain label=1: explicit Pride event, anti-racism program, ADA accommodation announcement.
          Examples of certain label=0: sports score, alumni fundraiser, class registration deadline.

- 80–94: You are confident but acknowledge minor ambiguity. The tweet leans clearly one way
          but contains some element that could cause a reasonable person to hesitate.
          Examples: HBCU mention without explicit DEI framing (lean 0 but unsure),
          women's leadership conference without equity language (lean 1 but borderline).

- 50–79: You are genuinely uncertain. The tweet could plausibly be either label.
          The context is limited, the framing is ambiguous, or a key word is present
          but without enough surrounding substance to confirm DEI intent.
          Use this range whenever you find yourself debating the label.

- 10–49: You assigned a label but acknowledge significant ambiguity.
          The tweet superficially resembles DEI content (or non-DEI) but lacks
          the substance to be confident. Flag these for human review.
          Example: "We welcome students of all backgrounds" — social-sounding but no DEI substance.

Key rule: A clearly non-DEI tweet (label=0) should get HIGH confidence (90+), not low.
Low confidence always means YOU are uncertain, not that the tweet is uncertain.
Do not default to 85. Use the full range.

RESPONSE FORMAT:
Think step by step. Then output ONLY one JSON object:
{"reasoning": "2-line explanation", "label": 0, "category": "campus_life", "confidence": 94}
Rules: reasoning = exactly 2 sentences. label = 0 or 1. category = one of 11 above. confidence = integer 0-100."""

VALID_CATEGORIES = {
    "lgbtq","race_ethnicity","disability","gender","general_dei",
    "social_justice","first_gen","admin","academic","campus_life","other",
}

# ── BitsAndBytesConfig ─────────────────────────────────────────────────────────
# float16 NOT bfloat16 — T4 has no native BF16 accelerator.
# BF16 on T4 silently upcasts to float32, doubling VRAM to ~14-15 GB (OOM).
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# ══════════════════════════════════════════════════════════════════════════════
# TOKENIZER
# ══════════════════════════════════════════════════════════════════════════════
_tok_config_path = os.path.join(SAVED_MODEL_PATH, "tokenizer_config.json")
if os.path.isfile(_tok_config_path):
    logger.info(f"Loading tokenizer from Drive saved path: {SAVED_MODEL_PATH}")
    judge_tokenizer = AutoTokenizer.from_pretrained(
        SAVED_MODEL_PATH, trust_remote_code=True
    )
else:
    # Downloads to LOCAL_HF_CACHE (local Colab disk) — fast, ~10 MB
    logger.info(f"Tokenizer not on Drive — downloading from HF to LOCAL cache: {JUDGE_PRIMARY}")
    judge_tokenizer = AutoTokenizer.from_pretrained(
        JUDGE_PRIMARY,
        trust_remote_code=True,
        cache_dir=LOCAL_HF_CACHE,   # <-- local Colab disk, NOT Drive
    )

if judge_tokenizer.pad_token is None:
    judge_tokenizer.pad_token = judge_tokenizer.eos_token
judge_tokenizer.padding_side = "left"
logger.info(f"Tokenizer ready | vocab_size={judge_tokenizer.vocab_size}")

# ══════════════════════════════════════════════════════════════════════════════
# MODEL LOAD — Two-path fallback
#
# PATH 1: Drive SAVED_MODEL_PATH has config.json (7-8 GB quantized)
#         → loads directly, ~2-3 min, no HF download, no local disk usage
#         → NORMAL PATH for all runs after the first
#
# PATH 2: HF download to LOCAL_HF_CACHE (Colab local disk, ephemeral)
#         → downloads 28 GB BF16 to /root/.cache (NOT Drive), quantizes in-place
#         → ~15-20 min, uses ~28 GB local disk (Colab has ~78 GB)
#         → sets _save_after_load=True so Block C-B saves the 7-8 GB result to Drive
#         → local BF16 cache is lost on disconnect — acceptable, Drive has quantized copy
#
# PATH 3: Fallback to Qwen2.5-14B-Instruct if Qwen3 unavailable
#         ⚠ WARNING: Qwen2.5 does NOT support enable_thinking=True
# ══════════════════════════════════════════════════════════════════════════════
JUDGE_MODEL_NAME = JUDGE_PRIMARY
_model_loaded    = False
_save_after_load = False   # True → Block C-B saves quantized weights to Drive

# ── Drive size helper ─────────────────────────────────────────────────────────
def _get_dir_size_gb(path: str) -> float:
    total = 0
    try:
        for dirpath, _, filenames in os.walk(path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                try: total += os.path.getsize(fp)
                except OSError: pass
    except Exception:
        pass
    return total / (1024**3)

_drive_size_before = _get_dir_size_gb(SAVED_MODEL_PATH)
logger.info(
    f"Drive saved path size BEFORE load: {_drive_size_before:.2f} GB "
    f"({'quantized model found — will use PATH 1' if _drive_size_before > 1.0 else 'EMPTY — will download to local disk then save quantized to Drive'})"
)

# ── PATH 1: Load quantized weights from Drive ─────────────────────────────────
_saved_config = os.path.join(SAVED_MODEL_PATH, "config.json")
if os.path.isfile(_saved_config) and _drive_size_before > 1.0:
    logger.info(
        f"PATH 1: Loading quantized model from Drive ({_drive_size_before:.1f} GB). "
        "No HF download, no local disk usage."
    )
    try:
        judge_model = AutoModelForCausalLM.from_pretrained(
            SAVED_MODEL_PATH,
            quantization_config=bnb_cfg,
            device_map="auto",
            trust_remote_code=True,
        )
        _model_loaded = True
        logger.info("PATH 1 success: quantized model loaded from Drive.")
    except Exception as _e:
        logger.warning(f"PATH 1 failed: {_e} | Falling back to HF download.")

# ── PATH 2/3: Download to LOCAL Colab disk (NOT Drive) ───────────────────────
if not _model_loaded:
    logger.info(
        "PATH 2: Downloading from HuggingFace → LOCAL Colab disk (/root/.cache).\n"
        "  28 GB BF16 download to local disk (~78 GB available). Drive is NOT used.\n"
        "  Block C-B will save only the 7-8 GB NF4 quantized result to Drive.\n"
        "  Expected time: ~15-20 min first run."
    )
    for _model_id in [JUDGE_PRIMARY, JUDGE_FALLBACK]:
        try:
            logger.info(
                f"Trying: {_model_id}\n"
                f"  BF16 download → LOCAL: {LOCAL_HF_CACHE}\n"
                f"  After load, quantized result → Drive: {SAVED_MODEL_PATH} (via Block C-B)"
            )
            judge_model = AutoModelForCausalLM.from_pretrained(
                _model_id,
                quantization_config=bnb_cfg,
                device_map="auto",
                trust_remote_code=True,
                cache_dir=LOCAL_HF_CACHE,   # <-- LOCAL disk, NOT Drive
            )
            JUDGE_MODEL_NAME = _model_id
            _model_loaded    = True
            _save_after_load = True          # Block C-B will save quantized to Drive

            if _model_id == JUDGE_FALLBACK:
                logger.warning(
                    "FALLBACK MODEL ACTIVE: Qwen2.5-14B-Instruct does NOT support "
                    "enable_thinking=True. In Block E-B, set enable_thinking=False in "
                    "apply_chat_template() or inference will break."
                )
            else:
                logger.info(f"PATH 2 success: {_model_id} downloaded to local disk.")
            break
        except Exception as _e:
            logger.warning(f"{_model_id} failed: {_e}")

if not _model_loaded:
    raise RuntimeError(
        "All model load paths failed. "
        "Check HF_TOKEN in Colab Secrets and Drive connection."
    )

judge_model.eval()
judge_model.config.use_cache = False

# ── VRAM stats ─────────────────────────────────────────────────────────────────
_vram_reserved = torch.cuda.memory_reserved(0)  / (1024**3)
_vram_alloc    = torch.cuda.memory_allocated(0) / (1024**3)
_vram_total    = torch.cuda.get_device_properties(0).total_memory / (1024**3)
_vram_pct      = _vram_reserved / _vram_total * 100
logger.info(
    f"VRAM after load: alloc={_vram_alloc:.1f} GB | reserved={_vram_reserved:.1f} GB | "
    f"total={_vram_total:.1f} GB | {_vram_pct:.0f}% reserved"
)
if _vram_pct > 85:
    logger.warning(
        f"VRAM reserved ({_vram_pct:.0f}%) is high. "
        "E-C's dynamic batch sizer will reduce MAX_BATCH_SIZE automatically. "
        "You can also manually lower MAX_BATCH_SIZE in Block E-C to 4-6."
    )

logger.info(f"Judge model     : {JUDGE_MODEL_NAME}")
logger.info(
    f"_save_after_load: {_save_after_load} "
    f"({'Block C-B WILL save 7-8 GB quantized weights to Drive' if _save_after_load else 'Block C-B will skip — quantized already on Drive'})"
)
logger.info(
    f"Drive usage note: BF16 download stayed on local Colab disk ({LOCAL_HF_CACHE}). "
    f"Drive will only grow by ~7-8 GB after Block C-B runs."
)
logger.info("Block C-A v5.0 complete. Run Block C-B v4.0.")

====++++====
Block C-A v5.0 Output
====++++====

2026-04-19 11:03:52,291 | INFO | HF BF16 cache (local Colab disk, ephemeral) → /root/.cache/huggingface
2026-04-19 11:03:52,293 | INFO | Quantized model path (Drive, permanent)     → /content/drive/MyDrive/Twitter/Finetune/judge/model_weights/qwen3_14b


/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


2026-04-19 11:03:53,874 | INFO | Loading tokenizer from Drive saved path: /content/drive/MyDrive/Twitter/Finetune/judge/model_weights/qwen3_14b
2026-04-19 11:03:56,632 | INFO | Tokenizer ready | vocab_size=151643
2026-04-19 11:03:56,647 | INFO | Drive saved path size BEFORE load: 9.26 GB (quantized model found — will use PATH 1)
2026-04-19 11:03:56,648 | INFO | PATH 1: Loading quantized model from Drive (9.3 GB). No HF download, no local disk usage.


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:212: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-19 11:07:43,972 | INFO | PATH 1 success: quantized model loaded from Drive.
2026-04-19 11:07:43,977 | INFO | VRAM after load: alloc=9.3 GB | reserved=9.3 GB | total=14.6 GB | 64% reserved
2026-04-19 11:07:43,979 | INFO | Judge model     : Qwen/Qwen3-14B
2026-04-19 11:07:43,980 | INFO | _save_after_load: False (Block C-B will skip — quantized already on Drive)
2026-04-19 11:07:43,981 | INFO | Drive usage note: BF16 download stayed on local Colab disk (/root/.cache/huggingface). Drive will only grow by ~7-8 GB after Block C-B runs.
2026-04-19 11:07:43,982 | INFO | Block C-A v5.0 complete. Run Block C-B v4.0.


In [ ]:
# ==========================
# Block C-B: Save Judge Model to Drive — Verify + Reload Test
# Version: v5.0
# Purpose: Save 4-bit quantized model to Drive named path, verify saved size,
#          run a one-forward-pass reload test to confirm future loads will work.
# ==========================
#
# What this block does: (~400 words)
#
# - Feature 1 (Conditional Save ✅): Only runs save_pretrained if _save_after_load
#   is True (set by C-A when HF download occurred). If the model was already loaded
#   from Drive (PATH 1), this block logs a skip message and exits immediately.
#   This means C-B is safe to re-run at any time — it will not overwrite a good
#   saved model with a potentially broken one from a failed session.
#
# - Feature 2 (Atomic Save with Size Verification ✅): save_pretrained writes
#   model weights (safetensors format) and tokenizer to SAVED_MODEL_PATH. After
#   the write completes, _get_dir_size_gb() is called to measure the actual bytes
#   written to Drive. Expected range: 7.0–9.0 GB for Qwen3-14B NF4 4-bit.
#   If the measured size is below 5.0 GB, something went wrong (partial write,
#   Drive mount error, permissions issue). A hard WARNING is logged and
#   _save_verified is set False so the reload test is skipped and the HF cache
#   (which is intact in LOCAL_HF_CACHE) is flagged as the backup load path.
#
# - Feature 3 (Reload Sanity Test ✅): After a verified save, loads the tokenizer
#   only from SAVED_MODEL_PATH and runs apply_chat_template on a one-sentence
#   dummy message. This tests that the saved tokenizer_config.json, vocab.json,
#   tokenizer.json, and special_tokens_map.json are all intact and readable. Does
#   NOT reload the full model (that would take 2-3 min and use double VRAM) —
#   the tokenizer round-trip is sufficient to confirm the directory structure is
#   valid for future loads.
#
# - Feature 4 (Shard Count Logging ✅): Counts the number of .safetensors files
#   in SAVED_MODEL_PATH after save. Expected: 8 shards for Qwen3-14B (same as
#   the HF source, but each shard is ~1 GB instead of ~4 GB because quantized).
#   If shard count is 0 or 1, the save almost certainly failed or produced an
#   incomplete output.
#
# - Feature 5 (Drive Size Diff ✅): Logs the difference between Drive size before
#   C-A loaded the model and after C-B saved it. This is the most intuitive
#   confirmation that the save worked: "Drive grew by 7.4 GB = save succeeded."
#   The before-size was captured in C-A as _drive_size_before.
#
# - Feature 6 (HF Cache Backup Notice ✅): Whether or not the named path save
#   succeeded, logs the LOCAL_HF_CACHE path and its current size. The HF cache
#   contains the full BF16 shards plus the hub metadata. It is a valid backup
#   load path (PATH 2 in C-A's fallback chain) but its hash-named structure
#   makes it harder to copy between accounts. Named path is always preferred.
#
# How it works:
# - Step 1: Guard — verify C-A globals are present.
# - Step 2: If _save_after_load is False, log skip and exit.
# - Step 3: Call judge_model.save_pretrained(SAVED_MODEL_PATH, safe_serialization=True).
# - Step 4: Call judge_tokenizer.save_pretrained(SAVED_MODEL_PATH).
# - Step 5: Measure saved size via _get_dir_size_gb(). Count .safetensors shards.
# - Step 6: If size >= 5 GB: run tokenizer reload test.
# - Step 7: Log final summary: saved GB, shard count, Drive delta, verification status.
#
# Solutions attempted that did not work:
# - Failed attempt 1: Using os.path.getsize on individual shard files immediately
#   after save_pretrained returns. Drive FUSE mount sometimes reports stale sizes
#   for 30-60 seconds after a large write (buffered flush). _get_dir_size_gb() with
#   os.walk is slower but reads the inode sizes which are updated after flush.
#   Adding a 5-second sleep before the size check resolves the stale read issue.
# - Failed attempt 2: Reloading the full model in C-B to verify. This would
#   require 14+ GB VRAM (existing model + reload attempt) causing OOM on T4.
#   Tokenizer-only reload is the right scope for a save verification test.
#
# Solutions implemented:
# - Solution 1: 5-second sleep after save before size check to allow Drive FUSE flush.
# - Solution 2: Tokenizer-only reload test instead of full model reload.
print("====++++====")
print("Block C-B v5.0 Output")
print("====++++====")
print()

import os, logging, time, glob
import torch

logging.raiseExceptions = False
logger = logging.getLogger("dei_glm")

# ── Guards ─────────────────────────────────────────────────────────────────────
# FIX: Using LOCAL_HF_CACHE instead of HF_CACHE_DIR to match C-A v5.0
_required_globals = [
    "judge_model", "judge_tokenizer", "SAVED_MODEL_PATH", "LOCAL_HF_CACHE",
    "_save_after_load", "JUDGE_MODEL_NAME", "_drive_size_before",
]
_missing = [g for g in _required_globals if g not in globals()]
if _missing:
    raise RuntimeError(f"Missing globals from Block C-A: {_missing}. Run C-A first.")

# ── Drive size helper (same as C-A) ───────────────────────────────────────────
def _get_dir_size_gb(path: str) -> float:
    total = 0
    try:
        for dirpath, _, filenames in os.walk(path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                try: total += os.path.getsize(fp)
                except OSError: pass
    except Exception:
        pass
    return total / (1024**3)

# ── Early exit: model was loaded from PATH 1 (Drive) — no save needed ─────────
if not _save_after_load:
    logger.info(
        "✅ C-B skip: _save_after_load=False. "
        "Model was loaded from Drive saved path (PATH 1). "
        f"Current Drive saved size: {_get_dir_size_gb(SAVED_MODEL_PATH):.2f} GB. "
        "No action needed."
    )
    print("✅ C-B skip: model already saved on Drive. Nothing to do.")
else:
    # ══════════════════════════════════════════════════════════════════════════
    # SAVE MODEL + TOKENIZER
    # ══════════════════════════════════════════════════════════════════════════
    logger.info(
        f"Saving 4-bit quantized model to: {SAVED_MODEL_PATH}\n"
        "  What is being saved: NF4 4-bit weights (~7-8 GB) — NOT the 28 GB BF16 source.\n"
        "  Once saved, all future runs (any account with Drive access) load from here.\n"
        "  Expected time: 2-4 min (Drive write speed dependent)."
    )
    _save_ok        = False
    _save_verified  = False
    _saved_size_gb  = 0.0
    _shard_count    = 0

    try:
        judge_model.save_pretrained(SAVED_MODEL_PATH, safe_serialization=True)
        judge_tokenizer.save_pretrained(SAVED_MODEL_PATH)
        _save_ok = True
        logger.info("save_pretrained() returned without error.")
    except Exception as _save_err:
        logger.warning(
            f"save_pretrained() raised: {_save_err}\n"
            f"Model is loaded correctly in VRAM. "
            f"HF cache at {LOCAL_HF_CACHE} is the backup load path for this session.\n"
            f"Fix the Drive issue and re-run C-B to save."
        )

    if _save_ok:
        # Allow Drive FUSE to flush buffered writes before reading sizes.
        logger.info("Waiting 5s for Drive FUSE flush before size check...")
        time.sleep(5)

        _saved_size_gb = _get_dir_size_gb(SAVED_MODEL_PATH)
        _shards        = glob.glob(os.path.join(SAVED_MODEL_PATH, "*.safetensors"))
        _shard_count   = len(_shards)
        _drive_delta   = _saved_size_gb - _drive_size_before

        logger.info(
            f"Drive size after save : {_saved_size_gb:.2f} GB\n"
            f"Shard files (.safetensors): {_shard_count}\n"
            f"Drive grew by: {_drive_delta:.2f} GB (expected ~7-8 GB)"
        )

        if _saved_size_gb < 5.0:
            logger.warning(
                f"⚠️ Saved size {_saved_size_gb:.2f} GB is below 5 GB threshold. "
                "Likely a partial write (Drive mount error or permissions). "
                f"HF cache at {LOCAL_HF_CACHE} is your backup. "
                "Do NOT delete the HF cache until you see a verified save >= 5 GB."
            )
        else:
            _save_verified = True
            logger.info(f"✅ Size check passed: {_saved_size_gb:.2f} GB >= 5 GB threshold.")

    # ── Tokenizer reload test ─────────────────────────────────────────────────
    if _save_verified:
        try:
            from transformers import AutoTokenizer as _ATok
            _test_tok = _ATok.from_pretrained(SAVED_MODEL_PATH, trust_remote_code=True)
            _dummy_prompt = _test_tok.apply_chat_template(
                [{"role": "user", "content": "hello"}],
                tokenize=False,
                add_generation_prompt=True,
            )
            assert len(_dummy_prompt) > 5
            del _test_tok, _dummy_prompt
            logger.info(
                "✅ Tokenizer reload test PASSED. "
                "SAVED_MODEL_PATH is confirmed valid for future PATH 1 loads.\n"
                "Next session (any account with Drive access): C-A will load "
                f"{_saved_size_gb:.1f} GB from Drive in ~2-3 min. No HF download needed."
            )
        except Exception as _test_err:
            logger.warning(
                f"Tokenizer reload test FAILED: {_test_err}\n"
                "The saved directory may be corrupt. "
                "Next run may fall through to HF download (PATH 2) again. "
                f"HF cache at {LOCAL_HF_CACHE} is the reliable backup."
            )

    # ── HF cache backup info ──────────────────────────────────────────────────
    _hf_cache_size = _get_dir_size_gb(LOCAL_HF_CACHE)
    logger.info(
        f"HF cache (backup PATH 2) at {LOCAL_HF_CACHE}: {_hf_cache_size:.1f} GB. "
        "Contains full BF16 shards. Harder to share between accounts (hash-named). "
        "Keep until SAVED_MODEL_PATH is verified."
    )

    # ── Final summary ─────────────────────────────────────────────────────────
    _status = "✅ SAVE VERIFIED" if _save_verified else ("⚠️ SAVE UNVERIFIED" if _save_ok else "❌ SAVE FAILED")
    logger.info(
        f"\n{'='*50}\n"
        f"Block C-B v5.0 Summary\n"
        f"  Status            : {_status}\n"
        f"  Model             : {JUDGE_MODEL_NAME}\n"
        f"  Saved path        : {SAVED_MODEL_PATH}\n"
        f"  Saved size        : {_saved_size_gb:.2f} GB\n"
        f"  Shard count       : {_shard_count}\n"
        f"  Drive grew by     : {_saved_size_gb - _drive_size_before:.2f} GB\n"
        f"  Reload test       : {'PASSED' if _save_verified else 'SKIPPED/FAILED'}\n"
        f"{'='*50}"
    )
    print(f"\n{_status} | {_saved_size_gb:.1f} GB on Drive | {_shard_count} shards")

logger.info("✅ Block C-B v5.0 complete.")

====++++====
Block C-B v5.0 Output
====++++====

2026-04-19 11:07:44,012 | INFO | ✅ C-B skip: _save_after_load=False. Model was loaded from Drive saved path (PATH 1). Current Drive saved size: 9.26 GB. No action needed.
✅ C-B skip: model already saved on Drive. Nothing to do.
2026-04-19 11:07:44,013 | INFO | ✅ Block C-B v5.0 complete.


In [ ]:
# ==========================
# Block D v2.0: classify_tweet() — unchanged logic, added _retry hint
# ==========================
print("====++++====\nBlock D v2.0 Output\n====++++====\n")

import json, logging, re, threading, time
import torch

logging.raiseExceptions = False
logger = logging.getLogger("dei_glm")

VALID_CATEGORIES = {
    "lgbtq","race_ethnicity","disability","gender","general_dei",
    "social_justice","first_gen","admin","academic","campus_life","other",
}

_CAT_ALIASES = {
    "institutional_dei":"general_dei","institutional":"general_dei","dei":"general_dei",
    "diversity_inclusion":"general_dei","diversity_and_inclusion":"general_dei",
    "diversity_equity_inclusion":"general_dei","general_diversity":"general_dei",
    "race":"race_ethnicity","racial_equity":"race_ethnicity","race_and_ethnicity":"race_ethnicity",
    "ethnicity":"race_ethnicity","lgbtq+":"lgbtq","queer":"lgbtq","transgender":"lgbtq",
    "gender_equity":"gender","women":"gender","womens_rights":"gender","feminism":"gender",
    "social_justice_advocacy":"social_justice","criminal_justice":"social_justice",
    "immigration":"social_justice","human_rights":"social_justice",
    "first_generation":"first_gen","first_gen_students":"first_gen","low_income":"first_gen",
    "administration":"admin","administrative":"admin","operations":"admin",
    "neurodiversity":"disability","accessibility":"disability",
}

def _extract_json(text):
    start = text.find('{')
    if start == -1:
        return None
    depth = 0
    for i, ch in enumerate(text[start:], start):
        if ch == '{':   depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None

def _strip_fences(text):
    text = re.sub(r'```(?:json|JSON)?\s*', '', text)
    return re.sub(r'```\s*', '', text).strip()

def _normalise_category(cat, tweet_idx):
    cat = cat.strip().lower().replace(" ", "_").replace("-", "_")
    if cat in VALID_CATEGORIES: return cat
    mapped = _CAT_ALIASES.get(cat)
    if mapped: return mapped
    for valid in VALID_CATEGORIES:
        if valid in cat or cat in valid:
            return valid
    return "other"

def _parse_response(raw_text, old_label, tweet_idx):
    try:
        if "</think>" in raw_text:
            post = raw_text.split("</think>", 1)[1].strip()
        elif "</think" in raw_text:
            post = raw_text.split("</think", 1)[1].strip()
            post = re.sub(r'^[^{]*', '', post)
        else:
            post = raw_text.strip()
        post      = _strip_fences(post)
        candidate = _extract_json(post)
        if not candidate:
            raise ValueError(f"No complete JSON block | post[:150]={post[:150]!r}")
        parsed = json.loads(candidate)
        label  = int(parsed.get("label", -1))
        if label not in (0, 1):
            raise ValueError(f"label={label} not in (0,1)")
        cat  = _normalise_category(str(parsed.get("category", "other")), tweet_idx)
        conf = max(0, min(100, int(parsed.get("confidence", 0))))
        return {
            "reasoning":  str(parsed.get("reasoning", "")).strip(),
            "label":      label,
            "category":   cat,
            "confidence": conf,
            "_parse_ok":  True,
        }
    except Exception as _e:
        logger.warning(f"idx={tweet_idx} PARSE_ERROR: {_e} | raw[:100]={raw_text[:100]!r}")
        return {
            "reasoning":  f"PARSE_ERROR: {raw_text[:100]}",
            "label":      old_label,   # fallback to original
            "category":   "other",
            "confidence": 0,
            "_parse_ok":  False,
        }

def classify_tweet(tweet, old_label, tweet_idx, judge_model, judge_tokenizer,
                   judge_system_prompt, timeout_secs=90, max_new_tokens=1024):
    _fallback = {"reasoning": f"TIMEOUT_OR_ERROR@{tweet_idx}", "label": old_label,
                 "category": "other", "confidence": 0, "_parse_ok": False}
    messages = [
        {"role": "system", "content": judge_system_prompt},
        {"role": "user",   "content": f"Tweet: {tweet}"},
    ]
    try:
        prompt    = judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc       = judge_tokenizer(prompt, return_tensors="pt").to(judge_model.device)
        input_len = enc["input_ids"].shape[1]
    except Exception as _e:
        logger.warning(f"idx={tweet_idx} tokenization error: {_e}")
        return _fallback
    _result, _error = [None], [None]
    def _generate():
        try:
            with torch.no_grad():
                out = judge_model.generate(
                    **enc, max_new_tokens=max_new_tokens, temperature=0.2,
                    do_sample=True, top_p=0.9, repetition_penalty=1.05,
                    pad_token_id=judge_tokenizer.eos_token_id,
                )
            _result[0] = judge_tokenizer.decode(out[0][input_len:], skip_special_tokens=False)
        except torch.cuda.OutOfMemoryError as _oom:
            import gc; gc.collect(); torch.cuda.empty_cache()
            _error[0] = f"OOM: {_oom}"
        except Exception as _e:
            _error[0] = str(_e)
    t = threading.Thread(target=_generate, daemon=True)
    t.start(); t.join(timeout=timeout_secs)
    if t.is_alive():
        logger.warning(f"idx={tweet_idx}: timeout after {timeout_secs}s.")
        return _fallback
    if _error[0]: logger.warning(f"idx={tweet_idx}: error: {_error[0]}"); return _fallback
    if _result[0] is None: return _fallback
    return _parse_response(_result[0], old_label, tweet_idx)

# Smoke test
_t = classify_tweet(
    tweet="Our university's Disability Resource Center is expanding accommodations for deaf students.",
    old_label=0, tweet_idx=-1, judge_model=judge_model,
    judge_tokenizer=judge_tokenizer, judge_system_prompt=JUDGE_SYSTEM_PROMPT,
)
print(f"Smoke test: {_t}")
assert _t["label"] in (0, 1)
logger.info("✅ Block D v2.0 complete.")

====++++====
Block D v2.0 Output
====++++====

Smoke test: {'reasoning': 'The tweet directly references disability accommodations, a core aspect of DEI. Expansion of services for deaf students aligns with disability rights and accessibility advocacy.', 'label': 1, 'category': 'disability', 'confidence': 95, '_parse_ok': True}
2026-04-19 11:08:15,135 | INFO | ✅ Block D v2.0 complete.


In [ ]:
# ==========================
# Block E-A-A: GLM Inference Engine — Setup, Helpers & Checkpoint Resume
# Version: v11.0
# Purpose: OOM cleanup, path constants, checkpoint resume, pending_df, and all
#          helper functions including the new per-tweet thinking extractor.
# ==========================
#
# What this block does: (~400 words)
#
# - Feature 1 (OOM Cleanup ✅): torch.cuda.synchronize() + gc.collect() +
#   empty_cache() clears VRAM fragments from prior cells. Logs alloc/reserved
#   before cleanup so VRAM state is visible in the log at startup.
#
# - Feature 2 (Path Constants ✅): CHECKPOINT, OUTPUT, REVIEW_PATH defined here
#   as the single source of truth. E-A-B, E-B, and E-C read these from globals().
#   F-A and F-B are now OPTIONAL — E-C saves OUTPUT directly after every batch.
#
# - Feature 3 (Checkpoint Resume ✅): Reads CHECKPOINT CSV if present, extracts
#   the _orig_idx=-999 metadata row to recover prior elapsed seconds, builds
#   done_indices set. pending_df = glm_df rows NOT in done_indices. All resume
#   logic lives here so E-C starts immediately on the pending slice.
#
# - Feature 4 (_extract_thinking_and_output ✅): Splits raw model output at
#   </think> into (thinking, output_section). Handles missing </think> gracefully
#   with a TRUNCATED marker.
#
# - Feature 5 (_extract_tweet_thinking NEW ✅): Given the shared batch thinking
#   text and a tweet's 0-based batch index, tries to extract the portion of the
#   thinking that discusses that specific tweet. Uses regex to find patterns like
#   "Tweet[0]:", "Tweet 0:", "id=0", "item 0". Falls back to returning the full
#   batch thinking if the pattern is not found — always shows something useful.
#   This solves the core complaint: all tweets showed identical shared thinking.
#   Now each tweet card shows its own extracted section AND the full blob below.
#
# - Feature 6 (_compute_dynamic_batch_size ✅): Estimates optimal batch size
#   from upcoming tweet lengths vs TOKEN_BUDGET and current VRAM pressure. Reads
#   MAX_BATCH_SIZE, MIN_BATCH_SIZE, TOKEN_BUDGET from E-C globals(). Defined here
#   so both E-A-B and E-C can call it without import.
#
# - Feature 7 (Rich Console ✅): Console(width=130) defined here and shared by
#   E-A-B and E-C via globals(). last_n_buffer deque(maxlen=5) initialised here.
#
# How it works:
# - Step 1: OOM cleanup.
# - Step 2: Define path constants and OUTPUT_COLS.
# - Step 3: Guard — verify glm_df in globals.
# - Step 4: Tag glm_df with _orig_idx if not present.
# - Step 5: Read CHECKPOINT → done_indices → pending_df.
# - Step 6: Define _extract_thinking_and_output, _extract_json_array,
#           _get_vram_stats, _extract_tweet_thinking, _compute_dynamic_batch_size.
# - Step 7: Print startup summary.
#
# Solutions attempted that did not work:
# - Failed attempt 1: Using sentence-boundary splitting to split batch thinking
#   into per-tweet sections. The model does not produce one sentence per tweet;
#   it writes paragraphs that reference multiple tweets. Regex on tweet index
#   markers is far more reliable.
# - Failed attempt 2: Keeping E-A as one block. At 512 lines it violates the
#   150-line code rule and makes the display builder impossible to find quickly.
#   Splitting into E-A-A (setup) + E-A-B (display) fixes both issues.
#
# Solutions implemented:
# - Solution 1: _extract_tweet_thinking() with multi-pattern regex + graceful fallback.
# - Solution 2: Block split at the display builder boundary.

print("====++++====")
print("Block E-A-A v11.0 Output")
print("====++++====")
print()

import os, gc, logging, time, re
from collections import deque
import pandas as pd
import numpy as np
import torch
from rich.console import Console
from rich.text    import Text
from rich.rule    import Rule

logging.raiseExceptions = False
logger = logging.getLogger("dei_glm")

# ── OOM Cleanup ───────────────────────────────────────────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if torch.cuda.is_available():
    try:    torch.cuda.synchronize()
    except: pass
    gc.collect()
    torch.cuda.empty_cache()
    _a = torch.cuda.memory_allocated(0)  / (1024**3)
    _r = torch.cuda.memory_reserved(0)   / (1024**3)
    logger.info(f"OOM cleanup. alloc={_a:.2f} GB  reserved={_r:.2f} GB")

# ── Path Constants — single source of truth ───────────────────────────────────
BASE_DIR    = "/content/drive/MyDrive/Twitter/Finetune"
CHECKPOINT  = os.path.join(BASE_DIR, "data/judge/glm_train_v3_4k_checkpoint.csv")
OUTPUT      = os.path.join(BASE_DIR, "data/processed/glm_train_v3_4k.csv")
REVIEW_PATH = os.path.join(BASE_DIR, "data/processed/glm_train_v3_4k_review.csv")
OUTPUT_COLS = [
    "Tweet", "old_label", "label", "reasoning", "category",
    "confidence", "fn_flag", "judge_agrees", "source", "_parse_ok", "thinking"
]

# ── Guard ─────────────────────────────────────────────────────────────────────
if "glm_df" not in globals():
    raise RuntimeError("glm_df not found. Run Block B v2.0 first.")

if "_orig_idx" not in glm_df.columns:
    glm_df = glm_df.copy()
    glm_df["_orig_idx"] = range(len(glm_df))
    logger.info(f"Tagged glm_df _orig_idx 0..{len(glm_df)-1}")

# ── Checkpoint Resume ─────────────────────────────────────────────────────────
done_df             = pd.DataFrame()
done_indices        = set()
_pipeline_start_ts  = time.time()
_prior_elapsed_sec  = 0.0

if os.path.isfile(CHECKPOINT):
    try:
        _raw = pd.read_csv(CHECKPOINT, encoding="utf-8", encoding_errors="replace")
        _meta = (_raw[_raw["_orig_idx"] == -999]
                 if "_orig_idx" in _raw.columns else pd.DataFrame())
        if len(_meta) > 0:
            try:
                _prior_elapsed_sec = float(_meta.iloc[0]["reasoning"])
                _pipeline_start_ts = time.time() - _prior_elapsed_sec
            except Exception:
                pass
        done_df = (_raw[_raw["_orig_idx"] != -999].copy()
                   if "_orig_idx" in _raw.columns else _raw)
        if "_orig_idx" in done_df.columns:
            done_indices = set(done_df["_orig_idx"].dropna().astype(int).tolist())
            logger.info(f"Resumed: {len(done_df):,} rows. Prior elapsed: {_prior_elapsed_sec:.0f}s.")
    except Exception as _e:
        logger.warning(f"Checkpoint load failed: {_e}. Starting fresh.")
        done_df = pd.DataFrame(); done_indices = set()
else:
    logger.info("No checkpoint — fresh run.")

pending_df    = glm_df[~glm_df["_orig_idx"].isin(done_indices)].reset_index(drop=True)
last_n_buffer = deque(maxlen=5)
console       = Console(width=130)
logger.info(f"Pending: {len(pending_df):,} | Done: {len(done_indices):,} | Total: {len(glm_df):,}")

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _extract_thinking_and_output(raw_text: str):
    if not raw_text:
        return None, ""
    if "<think>" in raw_text:
        t0 = raw_text.find("<think>") + len("<think>")
        if "</think>" in raw_text:
            t1         = raw_text.find("</think>")
            thinking   = raw_text[t0:t1].strip()
            output_sec = raw_text[t1 + len("</think>"):].strip()
        else:
            thinking   = raw_text[t0:].strip() + "\n⚠ [TRUNCATED — </think> never reached]"
            output_sec = ""
    else:
        thinking   = None
        output_sec = raw_text.strip()
    return thinking, output_sec


def _extract_json_array(text: str):
    start = text.find('[')
    if start == -1:
        return None
    depth = 0
    for i, ch in enumerate(text[start:], start):
        if ch == '[':   depth += 1
        elif ch == ']':
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None


def _get_vram_stats():
    if not torch.cuda.is_available():
        return 0.0, 0.0, 14.6
    props    = torch.cuda.get_device_properties(0)
    total    = props.total_memory / (1024**3)
    alloc    = torch.cuda.memory_allocated(0) / (1024**3)
    reserved = torch.cuda.memory_reserved(0)  / (1024**3)
    return alloc, reserved, total


def _extract_tweet_thinking(batch_thinking: str, tweet_batch_idx: int, batch_size: int) -> str:
    """
    Extract the portion of shared batch thinking relevant to one tweet.
    The model thinks about all tweets together inside one <think> block.
    We find the section that discusses tweet_batch_idx by matching common
    patterns the model uses: Tweet[0]: / Tweet 0: / id=0 / Starting with Tweet 0
    Falls back to the full batch thinking if extraction fails.
    """
    if not batch_thinking or batch_size <= 1:
        return batch_thinking or ""

    # Patterns that mark the START of discussion for this tweet index
    start_pats = [
        rf'[Tt]weet\s*\[?\s*{tweet_batch_idx}\s*\]?\s*[:\-]',
        rf'(?m)^[Tt]weet\s+{tweet_batch_idx}\b',
        rf'Starting with [Tt]weet\s+{tweet_batch_idx}\b',
        rf'\bid\s*[=:]\s*{tweet_batch_idx}\b',
        rf'#{tweet_batch_idx}\b[:\.]',
    ]
    # Patterns that mark the START of the NEXT tweet (= end of current section)
    next_idx   = tweet_batch_idx + 1
    end_pats   = [
        rf'[Tt]weet\s*\[?\s*{next_idx}\s*\]?\s*[:\-]',
        rf'(?m)^[Tt]weet\s+{next_idx}\b',
        rf'Starting with [Tt]weet\s+{next_idx}\b',
        rf'\bid\s*[=:]\s*{next_idx}\b',
        rf'#{next_idx}\b[:\.]',
    ] if tweet_batch_idx < batch_size - 1 else [
    # Last tweet — catch model's post-classification wrap-up
          r'(?i)Now,?\s+check',
          r'(?i)Let me (review|verify|check|go back)',
          r'(?i)checking the categor',
          r'(?i)Need to ensure',
          r'(?i)Also,?\s+confidence',
          r'(?i)looking back',
          r'(?i)In summary',
    ]

    start_pos = None
    for pat in start_pats:
        m = re.search(pat, batch_thinking)
        if m:
            start_pos = m.start()
            break

    if start_pos is None:
        return ""   # Caller will show full batch thinking as fallback

    end_pos = len(batch_thinking)
    for pat in end_pats:
        m = re.search(pat, batch_thinking[start_pos + 1:])
        if m:
            end_pos = start_pos + 1 + m.start()
            break

    extracted = batch_thinking[start_pos:end_pos].strip()
    return extracted if len(extracted) > 30 else ""


def _compute_dynamic_batch_size(upcoming_tweets: list,
                                vram_reserved_gb: float = 0.0,
                                vram_total_gb: float = 14.6) -> int:
    _max_bs = globals().get("MAX_BATCH_SIZE", 12)
    _min_bs = globals().get("MIN_BATCH_SIZE", 2)
    _budget = globals().get("TOKEN_BUDGET",   4096)
    if not upcoming_tweets:
        return _min_bs
    sample           = upcoming_tweets[:20]
    avg_chars        = sum(len(t) for t in sample) / len(sample)
    avg_tok_per_item = avg_chars / 3.5
    sys_tok          = 350
    overhead         = 30
    target           = _budget * 0.80
    tok_per_item     = avg_tok_per_item + overhead
    if tok_per_item <= 0:
        return _min_bs
    bs = int((target - sys_tok) / tok_per_item)
    bs = max(_min_bs, min(_max_bs, bs))
    vram_pct = (vram_reserved_gb / max(vram_total_gb, 0.1)) * 100
    if vram_pct > 85 and bs > 1:
        bs -= 1
        logger.info(f"VRAM tight ({vram_pct:.0f}% reserved) — batch size reduced to {bs}")
    return bs


# ── Startup summary ────────────────────────────────────────────────────────────
_ga, _gr, _gt  = _get_vram_stats()
_upcoming      = pending_df["Tweet"].iloc[:30].tolist() if len(pending_df) > 0 else []
BATCH_SIZE     = _compute_dynamic_batch_size(_upcoming, _gr, _gt)

console.rule("[bold yellow]GLM Inference Engine v11.0 — E-A-A Ready[/bold yellow]")
console.print(
    f"  Pending  : [bright_yellow]{len(pending_df):,}[/bright_yellow]  |  "
    f"Done : [bright_green]{len(done_indices):,}[/bright_green]  |  "
    f"Total : [white]{len(glm_df):,}[/white]\n"
    f"  Initial batch size : [bright_green]{BATCH_SIZE}[/bright_green]  "
    f"[dim](recomputed per-batch in E-C)[/dim]\n"
    f"  GPU alloc / reserved : [bright_cyan]{_ga:.2f} GB / {_gr:.2f} GB[/bright_cyan]\n"
    f"  [dim]Config (MAX_BATCH_SIZE etc.) defined in Block E-C.[/dim]",
    style="yellow"
)
console.print("✅ E-A-A v11.0 ready. Run Block E-A-B v11.0.", style="bold green")
logger.info(
    f"✅ Block E-A-A v11.0 | BATCH_SIZE={BATCH_SIZE} | "
    f"pending={len(pending_df):,} | done={len(done_indices):,} | "
    f"GPU alloc={_ga:.2f} GB  reserved={_gr:.2f} GB  total={_gt:.1f} GB"
)

====++++====
Block E-A-A v11.0 Output
====++++====

2026-04-19 11:08:15,569 | INFO | OOM cleanup. alloc=9.26 GB  reserved=9.31 GB
2026-04-19 11:08:15,571 | INFO | Tagged glm_df _orig_idx 0..5094
2026-04-19 11:08:16,363 | INFO | Resumed: 5,491 rows. Prior elapsed: 85873s.
2026-04-19 11:08:16,367 | INFO | Pending: 0 | Done: 5,095 | Total: 5,095


──────────────────────────────────────────── GLM Inference Engine v11.0 — E-A-A Ready ────────────────────────────────────────────

  Pending  : 0  |  Done : 5,095  |  Total : 5,095
  Initial batch size : 2  (recomputed per-batch in E-C)
  GPU alloc / reserved : 9.26 GB / 9.31 GB
  Config (MAX_BATCH_SIZE etc.) defined in Block E-C.

✅ E-A-A v11.0 ready. Run Block E-A-B v11.0.

2026-04-19 11:08:16,384 | INFO | ✅ Block E-A-A v11.0 | BATCH_SIZE=2 | pending=0 | done=5,095 | GPU alloc=9.26 GB  reserved=9.31 GB  total=14.6 GB


# E-B

In [ ]:
# ==========================
# Block E-A-B: GLM Inference Engine — Display Builder
# Version: v11.0
# Purpose: _build_glm_display() with per-tweet thinking extraction, no intermediate
#          JSON output phase in the live section, and batch-number tracking.
# ==========================
#
# What this block does: (~420 words)
#
# - Feature 1 (No Intermediate JSON Phase ✅): In v10.0 the live "GENERATING BATCH"
#   section had two sub-phases: "🧠 THINKING" (streaming <think> tokens) and
#   "📤 OUTPUTTING JSON" (streaming post-think tokens). The JSON phase showed raw
#   partial JSON like `center </think>  [   {"id": 0,` which was noise and
#   confused users into thinking it was an error. v11.0 removes the phase label
#   entirely — the live section always shows "🔴 GENERATING" with the streaming
#   thinking text. After </think> is detected in the stream, the phase text
#   simply says "Finalising JSON..." without showing the raw output.
#
# - Feature 2 (Per-Tweet Thinking Extraction ✅): In completed tweet boxes,
#   v10.0 showed the full shared batch thinking identically on every tweet card.
#   v11.0 calls _extract_tweet_thinking() to pull out the section of the thinking
#   that discusses this specific tweet (by batch index). Shows:
#     🔍 EXTRACTED THINKING FOR THIS TWEET:  [per-tweet section]
#     🧠 FULL BATCH THINKING:  [complete blob, no truncation]
#     💬 REASONING (JSON):  [2-line reasoning field]
#   If extraction fails (pattern not found), the "EXTRACTED" section says
#   "Could not isolate — see FULL BATCH THINKING below" and the full blob
#   is always shown. No information is ever hidden.
#
# - Feature 3 (Batch Number Tracking ✅): Each completed tweet entry now carries
#   a "batch_num" key. The header line shows "Batch N (Xs)" where N is the
#   sequential batch counter and X is the generation time for that batch.
#   This makes it easy to cross-reference tweets in the same batch.
#
# - Feature 4 (No Truncation ✅): v10.0 had a [:200] preview on live thinking
#   and the full thinking was shown only in completed boxes. v11.0 shows full
#   thinking always in completed boxes. Live preview shows last 800 chars of
#   the stream (enough to see what's happening without slowing the display).
#
# - Feature 5 (White Separator ✅): Separator between tweet boxes is style="white"
#   (was "grey15" = invisible on dark theme). Unchanged from v10.0.
#
# - Feature 6 (Session / Remaining ✅): Status bar unchanged from v10.0 fix.
#   Shows "Xm Ys / ETA" without redundant "Total pipeline" column.
#
# How it works:
# - Step 1: Guard — verify E-A-A globals are present.
# - Step 2: Import Rich components (already imported in E-A-A, safe to re-import).
# - Step 3: Define _build_glm_display() with all changes described above.
# - Step 4: Run a no-op display test to confirm the function renders.
#
# Solutions attempted that did not work:
# - Failed attempt 1: Hiding the JSON output phase by detecting </think> in the
#   live stream and switching to a spinner. TextIteratorStreamer yields tokens
#   one at a time — detecting the boundary and freezing the display mid-stream
#   caused the Rich Live context to stall. Solution: just show "Finalising..."
#   static text after </think> appears without trying to block the stream.
# - Failed attempt 2: Extracting per-tweet thinking in E-C at record time and
#   storing it in last_n_display. This required passing batch_idx down through
#   five layers of function calls. Cleaner to call _extract_tweet_thinking()
#   inside _build_glm_display() using the stored batch_size and batch_idx fields.
#
# Solutions implemented:
# - Solution 1: Removed JSON phase label. Live section always says "GENERATING".
#   After </think> detected, shows "Finalising JSON..." static line.
# - Solution 2: _extract_tweet_thinking() called inside _build_glm_display()
#   for each completed tweet, using entry["batch_idx"] (0-based batch position).

print("====++++====")
print("Block E-A-B v11.0 Output")
print("====++++====")
print()

import time, logging
from rich.console import Console
from rich.table   import Table
from rich.text    import Text
from rich.rule    import Rule
from rich         import box

logging.raiseExceptions = False
logger = logging.getLogger("dei_glm")

# ── Guard ─────────────────────────────────────────────────────────────────────
_required_eaa = [
    "console", "done_indices", "pending_df", "last_n_buffer",
    "_extract_tweet_thinking", "_compute_dynamic_batch_size",
    "_get_vram_stats", "CHECKPOINT", "OUTPUT_COLS",
]
_missing_eaa = [g for g in _required_eaa if g not in globals()]
if _missing_eaa:
    raise RuntimeError(f"Missing from E-A-A: {_missing_eaa}. Run Block E-A-A v11.0 first.")


def _build_glm_display(
    n_done, n_total, n_fn, n_fp, n_fail,
    session_elapsed, total_elapsed,
    ckpt_flag, ckpt_saved_count, ckpt_last_idx,
    gpu_alloc_gb, gpu_reserved_gb, gpu_total_gb,
    current_batch_size,
    next_batch_size=None,
    warning_log=None,
    last_n_tweets=None,
    live_thinking=None,
    live_tweets=None,
    gen_start_time=None,
    live_input_tokens=0,
) -> Table:

    # ── Derived ───────────────────────────────────────────────────────────────
    pct          = min(100.0, n_done / max(n_total, 1) * 100)
    filled       = max(0, int(44 * pct / 100))
    bar          = "█" * filled + "░" * (44 - filled)
    session_done = max(n_done - len(done_indices), 0)
    rate_min     = (session_done / session_elapsed * 60) if session_elapsed > 5 else 0.0
    rate_hr      = rate_min * 60
    eta_secs     = ((session_elapsed / max(session_done, 1)) * max(n_total - n_done, 0)
                    if session_done > 0 else 0)

    def _fmt(s):
        m, se = divmod(int(s), 60)
        h, m  = divmod(m, 60)
        return f"{h}h {m}m {se}s" if h else f"{m}m {se}s"

    def sl(label, val, style="white"):
        t = Text(no_wrap=True)
        t.append(f"  • {label:<30}", style="dim white")
        t.append(str(val), style=style)
        return t

    root = Table(box=None, show_header=False, padding=(0, 0), expand=True)
    root.add_column("c", no_wrap=True)
    SEP  = Text("═" * 130, style="grey30")
    sep2 = Text("─" * 130, style="grey23")

    # ── STATS BAR ──────────────────────────────────────────────────────────────
    root.add_row(sep2)
    bs_str = str(current_batch_size)
    if next_batch_size and next_batch_size != current_batch_size:
        bs_str += f"  →  next: {next_batch_size}"

    _max_bs = globals().get("MAX_BATCH_SIZE", 12)
    _mnt    = globals().get("MAX_NEW_TOKENS_PER_ITEM", 1024)
    _tb     = globals().get("TOKEN_BUDGET", 4096)
    _to     = globals().get("TIMEOUT_SECS", 60)
    _ce     = globals().get("CHECKPOINT_EVERY", 1)
    _en_th  = globals().get("ENABLE_THINKING", True)
    max_tok = min(current_batch_size * _mnt, 6144)

    root.add_row(sl("Batch size",          bs_str,
                    "bright_green" if current_batch_size >= 8 else "yellow"))
    root.add_row(sl("Output tokens cap",
                    f"{_mnt}/tweet  (batch total: {max_tok}  →  hard-capped at 6,144)"))
    root.add_row(sl("Thinking (COT)",
                    "ON — Qwen3 chain-of-thought" if _en_th else "OFF — faster, lower accuracy",
                    "bright_green" if _en_th else "yellow"))
    root.add_row(sl("Input token budget",  f"{_tb}  |  dynamic sizing: ON", "bright_cyan"))
    root.add_row(sl("Timeout / call",
                    f"{_to * current_batch_size}s  ({_to}s × {current_batch_size})"))
    root.add_row(sl("Done / Total",        f"{n_done:,} / {n_total:,}", "bright_yellow"))
    root.add_row(sl("FN rescued",          f"{n_fn}", "bold bright_green"))
    root.add_row(sl("FP flipped",          f"{n_fp}", "yellow"))
    root.add_row(sl("Errors",              f"{n_fail}",
                    "bold dark_orange" if n_fail else "bright_green"))
    root.add_row(sl("Rate",
                    f"{rate_min:.2f} tw/min  |  {rate_hr:.0f} tw/hr  |  ETA: {_fmt(eta_secs)}"
                    if rate_min > 0 else "calculating...",
                    "bright_cyan" if rate_min > 0 else "dim"))
    sess_str  = (f"{_fmt(session_elapsed)} / {_fmt(eta_secs)}"
                 if rate_min > 0 else f"{_fmt(session_elapsed)} / calculating...")
    root.add_row(sl("Session / Remaining", sess_str))
    gpu_pct   = (gpu_reserved_gb / max(gpu_total_gb, 0.1)) * 100
    gpu_col   = ("bold dark_orange" if gpu_pct > 90
                 else ("yellow" if gpu_pct > 82 else "bright_cyan"))
    root.add_row(sl("GPU VRAM (alloc/res/total)",
                    f"{gpu_alloc_gb:.2f} GB  /  {gpu_reserved_gb:.2f} GB  /  {gpu_total_gb:.1f} GB  "
                    f"({gpu_pct:.0f}% reserved)", gpu_col))
    until  = (_ce - (session_done % _ce) if session_done > 0 else _ce) if _ce > 0 else 1
    ck_str = (f"✅ saved  |  {ckpt_saved_count:,} rows on disk" if ckpt_flag
              else f"⏳ next in ~{until} batch(es)  |  {ckpt_saved_count:,} rows on disk")
    root.add_row(sl("Checkpoint", ck_str, "bright_green" if ckpt_flag else "dim"))
    root.add_row(sep2)

    # ── PROGRESS BAR ──────────────────────────────────────────────────────────
    prog = Text(no_wrap=True)
    prog.append("OVERALL    ", style="bold yellow")
    prog.append(f"{pct:5.1f}%  ", style="bold white")
    prog.append(bar, style="bright_green" if pct >= 100 else "bright_yellow")
    prog.append(f"  {n_done:,}/{n_total:,}", style="dim")
    root.add_row(SEP)
    root.add_row(prog)
    root.add_row(SEP)

    # ── LIVE BATCH SECTION ────────────────────────────────────────────────────
    # v11.0 FIX: No "📤 OUTPUTTING JSON" phase shown.
    # We show "🧠 THINKING" always. After </think> appears in stream, show
    # "Finalising JSON..." without exposing the raw partial JSON array.
    if live_tweets is not None:
        gen_elapsed_s = (time.time() - gen_start_time) if gen_start_time else 0
        in_json_phase = live_thinking and "</think>" in live_thinking

        hdr = Text(no_wrap=True)
        hdr.append(
            f"  🔴 GENERATING BATCH — {len(live_tweets)} tweets  |  "
            f"in: {live_input_tokens:,} tok  |  "
            f"out cap: {min(len(live_tweets) * _mnt, 6144):,} tok  |  "
            f"{gen_elapsed_s:.0f}s elapsed",
            style="bold bright_red")
        root.add_row(hdr)
        root.add_row(sep2)

        for bi, item in enumerate(live_tweets):
            tweet_str = item[0] if isinstance(item, (list, tuple)) else str(item)
            old_label = item[1] if isinstance(item, (list, tuple)) and len(item) > 1 else "?"
            orig_idx  = item[2] if isinstance(item, (list, tuple)) and len(item) > 2 else "?"
            disp      = tweet_str.replace("\n", " ").strip()
            if len(disp) > 90: disp = disp[:87] + "..."
            tw = Text(no_wrap=True)
            tw.append(f"  [{bi:2d}] ", style="dim white")
            tw.append(f"idx={orig_idx:<5}", style="bold white")
            tw.append(f"  old={old_label}  ", style="dim white")
            tw.append(f'"{disp}"', style="italic white")
            root.add_row(tw)

        root.add_row(sep2)

        if in_json_phase:
            # Hide raw JSON — show static status
            phase_txt = Text(no_wrap=True)
            phase_txt.append("  ⚙  Finalising JSON output...", style="bold bright_green")
            n_think_w = len(live_thinking.split("</think>")[0].split())
            phase_txt.append(f"  |  {n_think_w:,} words of thinking captured  |  "
                             f"{gen_elapsed_s:.1f}s", style="dim yellow")
            root.add_row(phase_txt)
        else:
            phase_txt = Text(no_wrap=True)
            n_out_w   = len(live_thinking.split()) if live_thinking else 0
            phase_txt.append("  🧠 THINKING", style="bold yellow")
            phase_txt.append(f"  |  ~{n_out_w:,} words  |  {gen_elapsed_s:.1f}s",
                             style="dim yellow")
            root.add_row(phase_txt)
            # Show last 800 chars of live thinking (preview, not full)
            if live_thinking:
                preview = live_thinking[-800:].replace("\n", " ")
                th = Text(no_wrap=False)
                th.append("  ", style="dim")
                th.append(preview, style="yellow")
                root.add_row(th)
            else:
                root.add_row(Text("  [waiting for first token...]", style="dim"))

        root.add_row(sep2)

    # ── COMPLETED TWEET BOXES ─────────────────────────────────────────────────
    if last_n_tweets:
        hdr = Text(no_wrap=True)
        hdr.append(f"  ✅ Last {len(last_n_tweets)} Completed Tweets (most recent first):",
                   style="bold bright_cyan")
        root.add_row(hdr)
        root.add_row(sep2)

        for entry in reversed(list(last_n_tweets)):
            idx         = entry.get("idx",         "?")
            tweet_text  = entry.get("tweet",        "")
            thinking    = entry.get("thinking",     None)
            result      = entry.get("result",       {})
            old_lbl     = entry.get("old_label",    "?")
            gen_ms      = entry.get("gen_ms",       0.0)
            batch_sz    = entry.get("batch_size",   1)
            batch_num   = entry.get("batch_num",    "?")
            batch_idx   = entry.get("batch_idx",    0)    # 0-based position in batch
            in_tok      = entry.get("input_tokens", 0)
            out_tok     = entry.get("output_tokens",0)
            parse_ok    = result.get("_parse_ok",   False)
            label       = result.get("label",       -1)
            conf        = result.get("confidence",  0)
            cat         = result.get("category",    "?")
            reasoning   = result.get("reasoning",   "")
            err_msg     = entry.get("error_msg",    "")
            gen_s       = gen_ms / 1000.0
            icon        = "✅" if parse_ok else "❌"
            label_col   = ("bright_green" if label == 1
                          else ("white" if label == 0 else "red"))

            # Header line
            h = Text(no_wrap=True)
            h.append(f"  {icon} ", style="bold bright_cyan" if parse_ok else "bold dark_orange")
            h.append(f"idx={idx:<6}", style="bold white")
            h.append(f"  Confidence: {conf}%", style="dim white")
            h.append(f"  Old: {old_lbl} → New: {label}", style=label_col)
            h.append(f"  Cat: {cat}", style="dim white")
            root.add_row(h)

            # Batch line
            tok_per_tweet = in_tok // max(batch_sz, 1)
            b_line = Text(no_wrap=True)
            b_line.append(f"     Batch: {batch_num} ({gen_s:.1f}s)", style="dim grey50")
            b_line.append(f"  |  in: {in_tok:,} tok ({tok_per_tweet}/tweet)", style="dim grey50")
            b_line.append(f"  |  out: {out_tok:,} tok", style="dim grey50")
            b_line.append(f"  |  batch_size={batch_sz}", style="dim grey50")
            root.add_row(b_line)

            # Tweet text
            tw = Text(no_wrap=False)
            tw.append("     📝 Tweet:\n     ", style="dim white")
            tw.append(tweet_text, style="white")
            root.add_row(tw)


            # Per-tweet extracted thinking only — no full batch thinking shown
            if thinking and "TRUNCATED" not in thinking:
                # extracted = _extract_tweet_thinking(thinking, batch_idx, batch_sz)
                # if extracted:
                ex_t = Text(no_wrap=False)
                ex_t.append("     🔍 THINKING:\n     ", style="bold yellow")
                ex_t.append(thinking, style="yellow")
                root.add_row(ex_t)


            elif thinking and "TRUNCATED" in thinking:
                root.add_row(Text("     🧠 ⚠ TRUNCATED — </think> never received",
                                  style="bold dark_orange"))
            # Per-tweet extracted thinking + Full batch thinking fallback




            # JSON Reasoning field
            if parse_ok and reasoning:
                rs = Text(no_wrap=False)
                rs.append("     💬 REASONING (JSON):\n     ", style="dim bright_green")
                rs.append(reasoning, style="bright_green")
                root.add_row(rs)
            elif not parse_ok:
                e = Text(no_wrap=False)
                e.append("     ❌ PARSE FAIL: ", style="bold dark_orange")
                e.append((err_msg or reasoning)[:200], style="dark_orange")
                root.add_row(e)

            root.add_row(Text("  " + "─" * 126, style="white"))

    # ── Warnings ──────────────────────────────────────────────────────────────
    if warning_log:
        root.add_row(sep2)
        root.add_row(Text(f"  ⚠  Recent warnings ({len(warning_log)} total):",
                         style="bold yellow"))
        for w in warning_log[-3:]:
            root.add_row(Text(f"    {w[:120]}", style="dim yellow", no_wrap=True))

    root.add_row(sep2)
    return root


# ── Smoke test ────────────────────────────────────────────────────────────────
_tbl = _build_glm_display(
    0, 100, 0, 0, 0, 0.0, 0.0,
    False, 0, -1, 9.3, 9.5, 14.6,
    current_batch_size=12,
)
console.print(_tbl)
console.print("✅ Block E-A-B v11.0 ready. Run Block E-B v6.4 → E-C v4.5.", style="bold green")
logger.info("✅ Block E-A-B v11.0 complete. _build_glm_display() updated with per-tweet thinking.")

====++++====
Block E-A-B v11.0 Output
====++++====



──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  • Batch size                    12                                                                                              
  • Output tokens cap             1024/tweet  (batch total: 6144  →  hard-capped at 6,144)                                        
  • Thinking (COT)                ON — Qwen3 chain-of-thought                                                                     
  • Input token budget            4096  |  dynamic sizing: ON                                                                     
  • Timeout / call                720s  (60s × 12)                                                                                
  • Done / Total                  0 / 100                                                                                         
  • FN rescued                    0                                                                                               
  • FP flipped                    0                                                                                               
  • Errors                        0                                                                                               
  • Rate                          calculating...                                                                                  
  • Session / Remaining           0m 0s / calculating...                                                                          
  • GPU VRAM (alloc/res/total)    9.30 GB  /  9.50 GB  /  14.6 GB  (65% reserved)                                                 
  • Checkpoint                    ⏳ next in ~1 batch(es)  |  0 rows on disk                                                      
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
OVERALL      0.0%  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  0/100                                                            
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

✅ Block E-A-B v11.0 ready. Run Block E-B v6.4 → E-C v4.5.

2026-04-19 11:08:16,439 | INFO | ✅ Block E-A-B v11.0 complete. _build_glm_display() updated with per-tweet thinking.


In [ ]:
# ==========================
# Block E-B: GLM Fast Classify Engine — Batch JSON + Streaming + Output Auto-Save
# Version: v6.4
# Purpose: All batch inference functions from v6.3 PLUS _flush_output_file() so
#          the final OUTPUT CSV is always current — F-A and F-B are now optional.
# ==========================
#
# What this block does: (~420 words)
#
# - Feature 1 (BATCH_JUDGE_SYSTEM_PROMPT ✅): Same as v6.3. Extends the single-tweet
#   JUDGE_SYSTEM_PROMPT with a === BATCH MODE === suffix instructing the model to
#   think through all N tweets together and emit a single JSON array. This is the
#   single source of truth — do not redefine in E-C.
#
# - Feature 2 (_classify_batch_streaming ✅): Same as v6.3. Builds messages,
#   tokenises, spawns a daemon thread for judge_model.generate(), and uses
#   TextIteratorStreamer to forward tokens to the Rich live display every 25 tokens.
#   Returns list of 7-tuples: (result, thinking, output_sec, err_msg, gen_ms,
#   input_tokens, output_tokens).
#
# - Feature 3 (classify_batch_adaptive ✅): Same as v6.3. Retry wrapper that
#   tries [n, 3, 2, 1] sub-batch sizes on OOM/TIMEOUT/parse failures.
#
# - Feature 4 (_flush_checkpoint_with_meta ✅ v6.3): Atomic checkpoint save with
#   os.makedirs() before every write (critical fix from v6.3). Unchanged.
#
# - Feature 5 (_flush_output_file NEW ✅): Atomically saves the final output CSV
#   to OUTPUT path on every call. Called from E-C after every checkpoint flush.
#   This makes F-B optional — the OUTPUT file is always current after each batch.
#   Uses the same .tmp → os.replace() atomic pattern as F-B.
#   Handles the edge case where done_df + new session rows may have duplicates:
#   deduplicates on _orig_idx (keep last) before writing, same as F-B does.
#   Does NOT delete the checkpoint (that remains F-B's job on clean completion).
#   If the write fails, logs a warning but does NOT raise — a failed output save
#   should never abort the inference loop; the checkpoint is the safety net.
#
# - Feature 6 (_cleanup_gpu ✅): Same as v6.3.
#
# - Feature 7 (ENABLE_THINKING toggle ✅ v6.4): ENABLE_THINKING is now read from
#   E-C globals() at call time rather than hardcoded. apply_chat_template() receives
#   enable_thinking=ENABLE_THINKING so toggling the E-C config takes effect without
#   rerunning E-B.
#
# How it works:
# - Step 1: Guard — verify all globals from A→D→E-A-A/B are present.
# - Step 2: Define BATCH_JUDGE_SYSTEM_PROMPT.
# - Step 3: Define helpers: _cleanup_gpu, _make_fallback, _strip_fences, _normalise_cat.
# - Step 4: Define _parse_batch_response.
# - Step 5: Define _classify_batch_streaming (reads ENABLE_THINKING from globals).
# - Step 6: Define classify_batch_adaptive (retry wrapper).
# - Step 7: Define _flush_checkpoint_with_meta (v6.3 makedirs fix).
# - Step 8: Define _flush_output_file (NEW — atomic output save).
#
# Solutions attempted that did not work:
# - Failed attempt 1: Saving output in a background thread after each batch.
#   Drive FUSE writes are not thread-safe; concurrent writes from the main thread
#   (checkpoint) and background thread (output) caused occasional corrupt CSV files.
#   Synchronous write after checkpoint (both in main thread sequentially) is safe.
# - Failed attempt 2: Having _flush_output_file raise on failure.
#   A failed Drive write mid-run would abort 30h of inference. Warn-and-continue
#   is the correct policy — the checkpoint always has the data.
#
# Solutions implemented:
# - Solution 1: Synchronous output flush in main thread, after checkpoint flush.
# - Solution 2: Warn-and-continue on output flush failure.
# - Solution 3: ENABLE_THINKING read from globals() at call time.

print("====++++====")
print("Block E-B v6.4 Output")
print("====++++====")
print()

import os, gc, logging, threading, json, re, time
import torch
import pandas as pd

logging.raiseExceptions = False
logger = logging.getLogger("dei_glm")

# Generation parameters — defaults; E-C overrides via globals
MAX_NEW_TOKENS_PER_ITEM = 1024
ENABLE_THINKING         = True
TIMEOUT_SECS            = 120
MEMORY_CLEANUP_EVERY    = 5

try:
    from transformers import TextIteratorStreamer
    _HAS_STREAMER = True
except ImportError:
    _HAS_STREAMER = False
    logger.warning("TextIteratorStreamer not available — live thinking display disabled.")

_required = [
    "judge_model", "judge_tokenizer", "JUDGE_SYSTEM_PROMPT",
    "CHECKPOINT", "OUTPUT", "OUTPUT_COLS", "done_df",
    "_parse_response", "_extract_thinking_and_output",
    "_extract_json_array", "_get_vram_stats", "BATCH_SIZE",
]
_missing = [r for r in _required if r not in globals()]
if _missing:
    raise RuntimeError(f"Missing globals: {_missing}. Run Blocks A→D→E-A-A→E-A-B first.")

# ══════════════════════════════════════════════════════════════════════════════
# BATCH SYSTEM PROMPT
# ══════════════════════════════════════════════════════════════════════════════
BATCH_JUDGE_SYSTEM_PROMPT = JUDGE_SYSTEM_PROMPT + """

=== BATCH MODE ===
You will receive multiple numbered tweets. Think about ALL of them together, then
output ONLY a single JSON array — one object per tweet, in the same order.
Required output format (no other text):
[
{"id": 0, "reasoning": "2 sentences.", "label": 0, "category": "admin", "confidence": 35},
{"id": 1, "reasoning": "2 sentences.", "label": 1, "category": "lgbtq",  "confidence": 96},
  ...
]
Rules:
- "id" must match the tweet number exactly (0-based).
- Output exactly as many objects as there are tweets.
- No markdown fences, no extra explanation outside the array.
- When thinking about each tweet, label it by its number: "Tweet 0:", "Tweet 1:", etc."""

# ══════════════════════════════════════════════════════════════════════════════
# LOW-LEVEL HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def _cleanup_gpu(sync: bool = True):
    if sync and torch.cuda.is_available():
        try:    torch.cuda.synchronize()
        except: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _make_fallback(old_label: int, tweet_idx: int, reason: str = "FALLBACK") -> dict:
    return {"reasoning": f"{reason}@{tweet_idx}", "label": old_label,
            "category": "other", "confidence": 0, "_parse_ok": False}

def _strip_fences(text: str) -> str:
    text = re.sub(r'```(?:json|JSON)?\s*', '', text)
    return re.sub(r'```\s*', '', text).strip()

_VALID_CATS = {
    "lgbtq","race_ethnicity","disability","gender","general_dei",
    "social_justice","first_gen","admin","academic","campus_life","other",
}
_CAT_ALIASES = {
    "institutional_dei":"general_dei","institutional":"general_dei","dei":"general_dei",
    "diversity_inclusion":"general_dei","diversity_and_inclusion":"general_dei",
    "diversity_equity_inclusion":"general_dei","general_diversity":"general_dei",
    "race":"race_ethnicity","racial_equity":"race_ethnicity",
    "race_and_ethnicity":"race_ethnicity","ethnicity":"race_ethnicity",
    "lgbtq+":"lgbtq","queer":"lgbtq","transgender":"lgbtq",
    "gender_equity":"gender","women":"gender","womens_rights":"gender","feminism":"gender",
    "social_justice_advocacy":"social_justice","criminal_justice":"social_justice",
    "immigration":"social_justice","human_rights":"social_justice",
    "first_generation":"first_gen","first_gen_students":"first_gen","low_income":"first_gen",
    "administration":"admin","administrative":"admin","operations":"admin",
    "neurodiversity":"disability","accessibility":"disability",
}

def _normalise_cat(cat: str) -> str:
    cat = cat.strip().lower().replace(" ","_").replace("-","_")
    if cat in _VALID_CATS:   return cat
    mapped = _CAT_ALIASES.get(cat)
    if mapped:               return mapped
    for v in _VALID_CATS:
        if v in cat or cat in v: return v
    return "other"

# ══════════════════════════════════════════════════════════════════════════════
# BATCH RESPONSE PARSER
# ══════════════════════════════════════════════════════════════════════════════
def _parse_batch_response(text: str, old_labels: list, indices: list):
    n    = len(old_labels)
    text = _strip_fences(text)
    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()
    arr_text = _extract_json_array(text)
    if not arr_text:
        err = "NO_JSON_ARRAY_FOUND"
        logger.warning(f"Batch parse: {err} | text[:150]={text[:150]!r}")
        return [(_make_fallback(ol, idx, err), err)
                for ol, idx in zip(old_labels, indices)]
    try:
        parsed_arr = json.loads(arr_text)
        if not isinstance(parsed_arr, list):
            parsed_arr = [parsed_arr]
    except json.JSONDecodeError as je:
        truncated = arr_text.rsplit(',', 1)[0] + ']'
        try:    parsed_arr = json.loads(truncated)
        except: return [(_make_fallback(ol, idx, f"JSON_ERR:{str(je)[:40]}"), str(je)[:40])
                        for ol, idx in zip(old_labels, indices)]
    results = []
    for i, (ol, idx) in enumerate(zip(old_labels, indices)):
        item = next((x for x in parsed_arr
                     if isinstance(x, dict) and x.get("id") == i), None)
        if item is None and i < len(parsed_arr):
            item = parsed_arr[i]
        if item and isinstance(item, dict):
            label = int(item.get("label", -1))
            if label not in (0, 1): label = ol
            cat   = _normalise_cat(str(item.get("category", "other")))
            conf  = max(0, min(100, int(item.get("confidence", 0))))
            results.append(({"reasoning": str(item.get("reasoning","")).strip(),
                              "label": label, "category": cat,
                              "confidence": conf, "_parse_ok": True}, ""))
        else:
            err = f"ITEM_{i}_MISSING"
            results.append((_make_fallback(ol, idx, err), err))
    return results

# ══════════════════════════════════════════════════════════════════════════════
# CORE BATCH CLASSIFY WITH STREAMING
# ══════════════════════════════════════════════════════════════════════════════
def _classify_batch_streaming(batch_items: list, live_update_fn=None):
    batch_size      = len(batch_items)
    tweets          = [str(item[0]) for item in batch_items]
    old_labels      = [int(item[1]) for item in batch_items]
    indices         = [int(item[2]) for item in batch_items]
    # Read config from E-C globals at call time so toggling ENABLE_THINKING takes effect
    _enable_thinking   = globals().get("ENABLE_THINKING", True)
    _max_new_tokens    = globals().get("MAX_NEW_TOKENS_PER_ITEM", 1024)
    _timeout           = globals().get("TIMEOUT_SECS", 120)
    max_new_tokens     = min(batch_size * _max_new_tokens, 6144)
    batch_timeout      = _timeout * max(batch_size, 1)

    if batch_size == 1:
        messages = [{"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                    {"role": "user",   "content": f"Tweet: {tweets[0]}"}]
    else:
        tweet_list = "\n".join([f'Tweet[{i}]: "{t[:450]}"' for i, t in enumerate(tweets)])
        messages   = [
            {"role": "system", "content": BATCH_JUDGE_SYSTEM_PROMPT},
            {"role": "user",   "content":
             f"Classify these {batch_size} tweets. "
             f"Output a JSON array with exactly {batch_size} objects (ids 0–{batch_size-1}).\n\n"
             f"{tweet_list}"},
        ]

    enc = None
    try:
        prompt = judge_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=_enable_thinking,
        )
        enc = judge_tokenizer(prompt, return_tensors="pt").to(judge_model.device)
    except Exception as _e:
        logger.warning(f"idx={indices} tokenize error: {_e}")
        err = f"TOKENIZE_ERR:{str(_e)[:60]}"
        return [(_make_fallback(ol, idx, err), None, "", err, 0.0, 0, 0)
                for ol, idx in zip(old_labels, indices)]

    input_tokens = int(enc["input_ids"].shape[1])

    if _HAS_STREAMER:
        streamer   = TextIteratorStreamer(judge_tokenizer, skip_special_tokens=False,
                                         skip_prompt=True, timeout=batch_timeout + 15)
        gen_kwargs = dict(**enc, max_new_tokens=max_new_tokens, temperature=0.6,
                          do_sample=True, top_p=0.95, repetition_penalty=1.05,
                          pad_token_id=judge_tokenizer.eos_token_id, streamer=streamer)
    else:
        streamer   = None
        gen_kwargs = dict(**enc, max_new_tokens=max_new_tokens, temperature=0.6,
                          do_sample=True, top_p=0.95, repetition_penalty=1.05,
                          pad_token_id=judge_tokenizer.eos_token_id)

    _gen_error, _gen_result = [None], [None]

    def _generate():
        try:
            with torch.no_grad():
                if streamer is not None:
                    judge_model.generate(**gen_kwargs)
                else:
                    out = judge_model.generate(**gen_kwargs)
                    _gen_result[0] = judge_tokenizer.decode(
                        out[0][enc["input_ids"].shape[1]:], skip_special_tokens=False)
                    del out
        except torch.cuda.OutOfMemoryError as _oom:
            _gen_error[0] = f"OOM:{str(_oom)[:100]}"
            try:    torch.cuda.synchronize()
            except: pass
            gc.collect(); torch.cuda.empty_cache()
        except Exception as _e:
            _gen_error[0] = str(_e)[:100]

    t_start    = time.time()
    gen_thread = threading.Thread(target=_generate, daemon=True)
    gen_thread.start()
    collected, timed_out = [], False

    if streamer is not None:
        try:
            for token in streamer:
                collected.append(token)
                if live_update_fn and len(collected) % 4 == 0:
                    live_update_fn("".join(collected), len(collected),
                                   (time.time() - t_start) * 1000)
        except Exception as _se:
            timed_out = True
    else:
        gen_thread.join(timeout=batch_timeout)
        if gen_thread.is_alive():
            timed_out = True
        elif _gen_result[0] is not None:
            collected = [_gen_result[0]]

    del enc
    gen_ms        = (time.time() - t_start) * 1000
    output_tokens = len(collected)
    gen_thread.join(timeout=3)

    if timed_out or gen_thread.is_alive():
        err = f"TIMEOUT after {batch_timeout}s"
        logger.warning(f"idx={indices}: {err}")
        _cleanup_gpu(sync=True)
        return [(_make_fallback(ol, idx, "TIMEOUT"), None, "", err,
                 gen_ms, input_tokens, output_tokens)
                for ol, idx in zip(old_labels, indices)]

    if _gen_error[0]:
        logger.warning(f"idx={indices}: {_gen_error[0]}")
        _cleanup_gpu(sync="OOM" in _gen_error[0])
        return [(_make_fallback(ol, idx, _gen_error[0][:40]), None, "",
                 _gen_error[0], gen_ms, input_tokens, output_tokens)
                for ol, idx in zip(old_labels, indices)]

    if not collected:
        _cleanup_gpu(sync=False)
        return [(_make_fallback(ol, idx, "NO_OUTPUT"), None, "",
                 "NO_OUTPUT", gen_ms, input_tokens, output_tokens)
                for ol, idx in zip(old_labels, indices)]

    raw_output           = "".join(collected)
    thinking, output_sec = _extract_thinking_and_output(raw_output)

    if batch_size == 1:
        parsed  = _parse_response(raw_output, old_labels[0], indices[0])
        err_msg = "" if parsed["_parse_ok"] else parsed["reasoning"][:80]
        return [(parsed, thinking, output_sec, err_msg, gen_ms, input_tokens, output_tokens)]
    else:
        src         = output_sec if output_sec else raw_output
        parsed_list = _parse_batch_response(src, old_labels, indices)
        return [(r, thinking, output_sec, e, gen_ms, input_tokens, output_tokens)
                for r, e in parsed_list]

# ══════════════════════════════════════════════════════════════════════════════
# ADAPTIVE WRAPPER
# ══════════════════════════════════════════════════════════════════════════════
def classify_batch_adaptive(batch_items: list, live_update_fn=None):
    n     = len(batch_items)
    sizes, seen = [], set()
    for s in [n, 3, 2, 1]:
        if 0 < s <= n and s not in seen:
            sizes.append(s); seen.add(s)

    last_results = None
    attempt_size = n
    for attempt_size in sizes:
        if attempt_size == n:
            sub_results = _classify_batch_streaming(batch_items, live_update_fn)
        else:
            sub_results = []
            for start in range(0, n, attempt_size):
                part = _classify_batch_streaming(batch_items[start:start+attempt_size],
                                                  live_update_fn)
                sub_results.extend(part)
        last_results = sub_results
        any_oom     = any("OOM"     in r[3] for r in sub_results)
        any_timeout = any("TIMEOUT" in r[3] for r in sub_results)
        all_ok      = all(r[0]["_parse_ok"] for r in sub_results)
        if any_oom or any_timeout:
            logger.warning(f"adaptive: size={attempt_size} OOM={any_oom} TIMEOUT={any_timeout}. Retry.")
            _cleanup_gpu(sync=True); continue
        if not all_ok:
            n_fail = sum(1 for r in sub_results if not r[0]["_parse_ok"])
            if attempt_size == 1:
                logger.warning(f"size=1: {n_fail} parse failures. Accepting."); break
            if n_fail <= n // 2:
                logger.info(f"size={attempt_size}: {n_fail}/{n} parse errors (<50%). Accepting."); break
            logger.warning(f"size={attempt_size}: {n_fail}/{n} parse errors. Trying smaller."); continue
        break
    return [(r[0], r[1], r[2], r[3], r[4], attempt_size, r[5], r[6])
            for r in last_results]

# ══════════════════════════════════════════════════════════════════════════════
# CHECKPOINT FLUSH  (v6.3 makedirs fix — unchanged)
# ══════════════════════════════════════════════════════════════════════════════
def _flush_checkpoint_with_meta(new_rows_batch: list, total_elapsed_sec: float,
                                 force: bool = False):
    global done_df
    if not new_rows_batch and not force:
        return False, len(done_df), -1
    try:
        _ckpt_dir = os.path.dirname(CHECKPOINT)
        if _ckpt_dir:
            os.makedirs(_ckpt_dir, exist_ok=True)
        batch_df = pd.DataFrame(new_rows_batch)
        done_df  = (pd.concat([done_df, batch_df], ignore_index=True)
                    if len(done_df) > 0 else batch_df)
        last_idx = (int(batch_df["_orig_idx"].iloc[-1])
                    if "_orig_idx" in batch_df.columns and len(batch_df) > 0 else -1)
        meta_row = pd.DataFrame([{
            "Tweet":"__META__","old_label":-1,"label":-1,
            "reasoning":str(total_elapsed_sec),"category":"__meta__",
            "confidence":-1,"fn_flag":-1,"judge_agrees":-1,
            "source":"__meta__","_parse_ok":False,"_orig_idx":-999,
        }])
        ckpt_with_meta = pd.concat([meta_row, done_df], ignore_index=True)
        _tmp = CHECKPOINT + ".tmp"
        ckpt_with_meta.to_csv(_tmp, index=False, encoding="utf-8")
        os.replace(_tmp, CHECKPOINT)
        n_saved = len(done_df)
        logger.info(f"Checkpoint: {n_saved:,} rows → {CHECKPOINT} | last_idx={last_idx}")
        return True, n_saved, last_idx
    except Exception as _e:
        logger.warning(f"Checkpoint write failed: {_e} — in-memory state preserved.")
        return False, len(done_df), -1

# ══════════════════════════════════════════════════════════════════════════════
# OUTPUT FILE FLUSH  (v6.4 NEW — makes F-A/F-B optional)
# ══════════════════════════════════════════════════════════════════════════════
def _flush_output_file(new_rows: list) -> bool:
    """
    Atomically write the final OUTPUT CSV from done_df + new_rows (this session).
    Called after every batch. F-B is now optional — OUTPUT is always current.
    Never raises — a failed write logs a warning but does not abort inference.

    Deduplication logic (mirrors F-B):
      concat done_df + session rows → drop _orig_idx=-999 meta rows →
      deduplicate on _orig_idx (keep last) → sort → drop _orig_idx →
      write OUTPUT_COLS only.
    """
    global done_df
    try:
        _out_dir = os.path.dirname(OUTPUT)
        if _out_dir:
            os.makedirs(_out_dir, exist_ok=True)

        _session_df = pd.DataFrame(new_rows) if new_rows else pd.DataFrame()
        _all = (pd.concat([done_df, _session_df], ignore_index=True)
                if len(done_df) > 0 else _session_df)

        if "_orig_idx" in _all.columns:
            _all = (_all[_all["_orig_idx"] != -999]
                    .drop_duplicates(subset=["_orig_idx"], keep="last")
                    .sort_values("_orig_idx")
                    .reset_index(drop=True))

        # Write only OUTPUT_COLS that exist (graceful if a col is missing)
        _cols = [c for c in OUTPUT_COLS if c in _all.columns]
        _tmp  = OUTPUT + ".tmp"
        _all[_cols].to_csv(_tmp, index=False, encoding="utf-8")
        os.replace(_tmp, OUTPUT)
        logger.info(f"Output file updated: {len(_all):,} rows → {OUTPUT}")
        return True
    except Exception as _e:
        logger.warning(f"Output file flush failed: {_e}  (checkpoint is your safety net)")
        try:
            if os.path.exists(OUTPUT + ".tmp"):
                os.remove(OUTPUT + ".tmp")
        except Exception:
            pass
        return False


logger.info(
    "✅ Block E-B v6.4: all functions defined.\n"
    "  NEW: _flush_output_file() saves OUTPUT after every batch — F-A/F-B now optional.\n"
    "  FIX: ENABLE_THINKING read from E-C globals() at call time."
)
print("✅ E-B v6.4 ready. Run Block E-C v4.5 to start inference loop.")

====++++====
Block E-B v6.4 Output
====++++====

2026-04-19 11:08:16,501 | INFO | ✅ Block E-B v6.4: all functions defined.
  NEW: _flush_output_file() saves OUTPUT after every batch — F-A/F-B now optional.
  FIX: ENABLE_THINKING read from E-C globals() at call time.
✅ E-B v6.4 ready. Run Block E-C v4.5 to start inference loop.


In [ ]:
# ==========================
# Block E-C: GLM Inference Loop — Batch JSON + Live Display + Auto-Save
# Version: v4.5
# Purpose: Config constants, main inference loop, auto-save OUTPUT after every batch,
#          atexit/signal handler so data survives session death, F-A/F-B optional.
# ==========================
#
# What this block does: (~440 words)
#
# - Feature 1 (Single Source of Truth Config ✅): ALL numeric constants here.
#   ENABLE_THINKING is now a first-class config with a clear comment explaining
#   the speed/quality tradeoff. Setting it False gives 2-4× speedup at the cost
#   of missing borderline DEI cases. Recommended: True for final run, False for
#   a fast preview pass.
#
# - Feature 2 (Auto-Save OUTPUT After Every Batch ✅ v4.5 NEW): After each
#   checkpoint flush, calls _flush_output_file(new_rows) to write the complete
#   OUTPUT CSV. This means at any point in time, OUTPUT reflects all completed
#   rows. F-A and F-B are now truly optional — they only add the analysis display
#   and the review file export. The checkpoint is the resume mechanism; OUTPUT is
#   the deliverable.
#
# - Feature 3 (atexit + SIGTERM Handler ✅ v4.5 NEW): Registers an atexit handler
#   and a SIGTERM signal handler that both call _emergency_save(). This fires when:
#     - Colab runtime is killed (SIGTERM)
#     - Python exits normally after a KeyboardInterrupt
#     - Session timeout
#   _emergency_save() flushes batch_buffer to checkpoint AND calls _flush_output_file().
#   Data loss window reduced from "entire session" to "current in-progress batch".
#
# - Feature 4 (Batch Number Tracking ✅ v4.5 NEW): _batch_counter increments on
#   every batch. Passed to last_n_display entries as "batch_num". Display shows
#   "Batch 14 (23.4s)" per tweet card so you can see which batch produced a result.
#
# - Feature 5 (batch_idx Tracking ✅ v4.5 NEW): Each result entry stores
#   "batch_idx" (0-based position within its batch). _build_glm_display() uses
#   this to call _extract_tweet_thinking(thinking, batch_idx, batch_sz), extracting
#   the per-tweet section of the shared batch thinking.
#
# - Feature 6 (Speed Options Comment ✅): Clearly documents all speed levers:
#     ENABLE_THINKING=False  → 2-4× faster, lower accuracy
#     MAX_NEW_TOKENS=512     → ~1.5× faster (less output space)
#     MAX_BATCH_SIZE=8       → less OOM risk, slightly lower throughput
#   Note: 69% VRAM reserved ≠ 69% compute utilization. During generate(), GPU
#   compute is near 100%. The "idle" periods are tokenization + checkpoint I/O.
#   The only way to meaningfully speed up generation itself is fewer output tokens
#   (thinking=False or lower max_new_tokens).
#
# - Feature 7 (final_df Assembly ✅): Same as v4.4. Assembles final_df from
#   done_df + new_rows, deduplicated and sorted. Available after loop for F-A/F-B.
#
# How it works:
# - Step 1: Guard — all required globals.
# - Step 2: Config constants.
# - Step 3: Register atexit + SIGTERM handlers.
# - Step 4: WarningCollector, batch counter init, source lookup dict.
# - Step 5: Enter Rich Live.
# - Step 6: Iterate → classify → record (with batch_num + batch_idx) → flush
#           checkpoint → flush OUTPUT → update display.
# - Step 7: Final flush in finally + assemble final_df.
#
# Solutions attempted that did not work:
# - Failed attempt 1: Using SIGKILL handler. SIGKILL cannot be caught in Python
#   (or any language). Only SIGTERM can be caught. Colab sends SIGTERM before
#   SIGKILL so we catch SIGTERM for the graceful window.
# - Failed attempt 2: asyncio-based checkpoint save. Event loop conflicts with
#   PyTorch CUDA streams and TextIteratorStreamer's internal threading caused
#   deadlocks. Synchronous saves in the main thread after each batch is correct.
#
# Solutions implemented:
# - Solution 1: atexit.register() + signal.signal(SIGTERM) both call _emergency_save().
# - Solution 2: Batch counter and batch_idx stored in every last_n_display entry.
# - Solution 3: _flush_output_file() called after every _flush_checkpoint_with_meta().

print("====++++====")
print("Block E-C v4.5 Output")
print("====++++====")
print()

import os, gc, logging, time, atexit, signal
import pandas as pd
import torch
from rich.live  import Live
from rich.text  import Text
from rich       import box
from collections import deque

logging.raiseExceptions = False
logger = logging.getLogger("dei_glm")

# ── Guards ─────────────────────────────────────────────────────────────────────
_required = [
    "pending_df","done_df","done_indices","glm_df",
    "judge_model","judge_tokenizer","JUDGE_SYSTEM_PROMPT",
    "_build_glm_display","_extract_thinking_and_output","console",
    "CHECKPOINT","OUTPUT","OUTPUT_COLS","last_n_buffer",
    "classify_batch_adaptive","_flush_checkpoint_with_meta",
    "_flush_output_file","_cleanup_gpu","_pipeline_start_ts",
    "_prior_elapsed_sec","BATCH_SIZE","_get_vram_stats",
    "_make_fallback","_compute_dynamic_batch_size","_extract_tweet_thinking",
]
_missing = [r for r in _required if r not in globals()]
if _missing:
    raise RuntimeError(f"Missing: {_missing}. Run A→B→C→D→E-A-A→E-A-B→E-B v6.4 first.")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — SINGLE SOURCE OF TRUTH
# ══════════════════════════════════════════════════════════════════════════════

# ─── SPEED / QUALITY TRADEOFF ──────────────────────────────────────────────────
# ENABLE_THINKING = True   → Qwen3 chain-of-thought ON
#                            Model generates a <think> block before each JSON answer.
#                            Higher accuracy on borderline tweets. ~2-4× SLOWER.
#                            Recommended for final training data production.
#
# ENABLE_THINKING = False  → No <think> block. Direct JSON output.
#                            ~2-4× FASTER. Misses subtle/borderline DEI cases.
#                            Recommended for a fast preview pass or re-labelling
#                            high-confidence tweets you already trust.
#
# If you are at 69% VRAM and want more speed: disable thinking first.
# Your GPU compute IS near 100% during generation — the "idle" is Python overhead.
# ────────────────────────────────────────────────────────────────────────────────
ENABLE_THINKING         = True    # ← Toggle here. Read by E-B at call time.

MAX_BATCH_SIZE          = 12      # Hard cap. T4 handles 12 with COT ON, 16 with COT OFF.
MIN_BATCH_SIZE          = 2       # Floor for adaptive sizing.
TOKEN_BUDGET            = 4096    # Input token target for dynamic batch calc.
MAX_NEW_TOKENS_PER_ITEM = 1024    # Per tweet. Batch total hard-capped at 6144.
                                   # Reduce to 512 for ~1.5× speedup (less COT depth).
TIMEOUT_SECS            = 60      # Per tweet. Actual batch timeout = this × batch_size.
CHECKPOINT_EVERY        = 1       # Flush after every N batches (1 = after every batch).
LAST_N_DISPLAY          = 10       # Completed tweet boxes in live display.
MEMORY_CLEANUP_EVERY    = 1       # GPU cleanup frequency (batches).

logger.info(
    f"Block E-C v4.5 | pending={len(pending_df):,} | COT={ENABLE_THINKING} | "
    f"max_batch_size={MAX_BATCH_SIZE} | max_tokens/item={MAX_NEW_TOKENS_PER_ITEM} | "
    f"timeout={TIMEOUT_SECS}s | ckpt_every=EVERY BATCH | output_every=EVERY BATCH | v4.5"
)

# ══════════════════════════════════════════════════════════════════════════════
# ATEXIT + SIGTERM EMERGENCY SAVE  (v4.5 NEW)
# Fires when: Colab kills session (SIGTERM), KeyboardInterrupt exits Python,
# session timeout. Saves checkpoint + output file.
# ══════════════════════════════════════════════════════════════════════════════
_batch_buffer_ref   = []   # Mutable ref so atexit handler sees current state
_new_rows_ref       = []   # Same for new_rows
_emergency_called   = False

def _emergency_save():
    global _emergency_called
    if _emergency_called:
        return
    _emergency_called = True
    try:
        _total_now = time.time() - _pipeline_start_ts
        if _batch_buffer_ref:
            logger.warning(f"Emergency save: flushing {len(_batch_buffer_ref)} buffered rows.")
            _flush_checkpoint_with_meta(_batch_buffer_ref, _total_now, force=True)
            _batch_buffer_ref.clear()
        _flush_output_file(_new_rows_ref)
        logger.info("Emergency save complete.")
    except Exception as _ee:
        logger.error(f"Emergency save failed: {_ee}")

atexit.register(_emergency_save)

def _sigterm_handler(signum, frame):
    logger.warning("SIGTERM received — triggering emergency save.")
    _emergency_save()

try:
    signal.signal(signal.SIGTERM, _sigterm_handler)
except Exception:
    pass   # SIGTERM may not be available in all environments

# ── Warning collector ──────────────────────────────────────────────────────────
class _WarningCollector(logging.Handler):
    def __init__(self):
        super().__init__(level=logging.WARNING)
        self.records = []
    def emit(self, record):
        self.records.append(self.format(record))

_warn_collector = _WarningCollector()
_warn_collector.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
_stream_handler = None
for _h in list(logger.handlers):
    if isinstance(_h, logging.StreamHandler) and not isinstance(_h, logging.FileHandler):
        _stream_handler = _h
        logger.removeHandler(_h)
        break
logger.addHandler(_warn_collector)

# ── O(1) source lookup ─────────────────────────────────────────────────────────
_orig_idx_to_source = dict(
    zip(pending_df["_orig_idx"].astype(int), pending_df["source"].astype(str))
)

# ══════════════════════════════════════════════════════════════════════════════
# RUNTIME STATE
# ══════════════════════════════════════════════════════════════════════════════
new_rows         = []
_new_rows_ref    = new_rows   # point emergency handler at the live list
n_done           = len(done_indices)
n_fn             = (int(done_df["fn_flag"].sum())
                    if "fn_flag" in done_df.columns and len(done_df) > 0 else 0)
n_fp             = (int(((done_df.get("old_label", pd.Series(dtype=int)) == 1) &
                         (done_df.get("label",     pd.Series(dtype=int)) == 0)).sum())
                    if len(done_df) > 0 else 0)
n_fail           = (int((~done_df["_parse_ok"]).sum())
                    if "_parse_ok" in done_df.columns and len(done_df) > 0 else 0)
session_start    = time.time()
n_total          = len(glm_df)
_ga, _gr, _gt   = _get_vram_stats()
ckpt_saved_count = len(done_df)
ckpt_last_idx    = -1
ckpt_flag        = False
batch_buffer     = []
_batch_buffer_ref = batch_buffer   # emergency handler ref
last_n_display   = deque(maxlen=LAST_N_DISPLAY)
_batch_counter   = 0

_stream_state = {
    "live_thinking":     None,
    "live_tweets":       None,
    "gen_start_time":    None,
    "live_input_tokens": 0,
}

# ── Dynamic batch generator ────────────────────────────────────────────────────
def _make_batches_dynamic(df, max_bs=MAX_BATCH_SIZE):
    items = [(str(r["Tweet"]), int(r["old_label"]), int(r["_orig_idx"]))
             for _, r in df.iterrows()]
    i = 0
    while i < len(items):
        look_ahead = [it[0] for it in items[i : i + max_bs * 2]]
        _a, _r, _t = _get_vram_stats()
        bs = _compute_dynamic_batch_size(look_ahead, _r, _t)
        yield items[i : i + bs]
        i += bs

# ══════════════════════════════════════════════════════════════════════════════
# MAIN INFERENCE LOOP
# ══════════════════════════════════════════════════════════════════════════════
_stop = False
try:
    _ga, _gr, _gt = _get_vram_stats()
    with Live(
        _build_glm_display(
            n_done, n_total, n_fn, n_fp, n_fail, 0.0, 0.0,
            False, ckpt_saved_count, ckpt_last_idx,
            _ga, _gr, _gt, current_batch_size=BATCH_SIZE,
            warning_log=_warn_collector.records,
            last_n_tweets=list(last_n_display),
        ),
        console=console, refresh_per_second=4, transient=False,
    ) as live:

        def _upd(actual_bs=BATCH_SIZE, next_bs=None):
            _sess = time.time() - session_start
            _tot  = time.time() - _pipeline_start_ts
            _a, _r, _t = _get_vram_stats()
            live.update(_build_glm_display(
                n_done, n_total, n_fn, n_fp, n_fail, _sess, _tot,
                ckpt_flag, ckpt_saved_count, ckpt_last_idx,
                _a, _r, _t,
                current_batch_size=actual_bs, next_batch_size=next_bs,
                warning_log=_warn_collector.records,
                last_n_tweets=list(last_n_display),
                live_thinking=_stream_state["live_thinking"],
                live_tweets=_stream_state["live_tweets"],
                gen_start_time=_stream_state["gen_start_time"],
                live_input_tokens=_stream_state["live_input_tokens"],
            ))

        def _live_cb(text, n_tokens, elapsed_ms):
            _stream_state["live_thinking"] = text
            if n_tokens % 4 == 0:
                _upd()

        _row_cursor = 0
        for batch in _make_batches_dynamic(pending_df, MAX_BATCH_SIZE):
            if _stop:
                break
            _batch_counter += 1
            _stream_state["live_thinking"]     = ""
            _stream_state["live_tweets"]       = batch
            _stream_state["gen_start_time"]    = time.time()
            _stream_state["live_input_tokens"] = 0
            _upd(len(batch))

            # ── Classify ──────────────────────────────────────────────────────
            try:
                results = classify_batch_adaptive(batch, live_update_fn=_live_cb)
            except Exception as _oe:
                logger.warning(f"Outer batch error: {_oe}")
                results = [
                    (_make_fallback(ol, idx, f"OUTER_ERR:{str(_oe)[:40]}"),
                     None, "", str(_oe)[:80], 0.0, 1, 0, 0)
                    for _, ol, idx in batch
                ]

            _stream_state["live_thinking"]  = None
            _stream_state["live_tweets"]    = None
            _stream_state["gen_start_time"] = None
            _cleanup_gpu(sync=False)

            # ── Record results ─────────────────────────────────────────────────
            actual_bs_used      = results[0][5] if results else 1
            _row_cursor        += actual_bs_used
            batch_input_tokens  = results[0][6] if results else 0
            batch_output_tokens = results[0][7] if results else 0

            for i, item in enumerate(results):
                result, thinking, output_sec, err_msg, gen_ms, actual_bs, in_tok, out_tok = item
                tweet_thinking = _extract_tweet_thinking(thinking, i, actual_bs)
                if not tweet_thinking:
                    tweet_thinking = thinking
                tweet, old_label, orig_idx = batch[i]
                source       = _orig_idx_to_source.get(orig_idx, "unknown")
                new_label    = result["label"]
                fn_flag      = 1 if (old_label == 0 and new_label == 1) else 0
                judge_agrees = 1 if old_label == new_label else 0

                result_row = {
                    "Tweet": tweet, "old_label": old_label, "label": new_label,
                    "reasoning": result["reasoning"], "category": result["category"],
                    "confidence": result["confidence"], "fn_flag": fn_flag,
                    "judge_agrees": judge_agrees, "source": source,
                    "_parse_ok": result["_parse_ok"], "_orig_idx": orig_idx,
                    "thinking": tweet_thinking,
                }
                batch_buffer.append(result_row)
                new_rows.append(result_row)

                last_n_display.append({
                    "idx":          orig_idx,
                    "tweet":        tweet,
                    "old_label":    old_label,
                    "thinking":     tweet_thinking,
                    "output_sec":   output_sec,
                    "result":       result,
                    "error_msg":    err_msg,
                    "gen_ms":       gen_ms / max(actual_bs, 1),
                    "batch_size":   actual_bs,
                    "batch_num":    _batch_counter,          # v4.5: batch number
                    "batch_idx":    i,                       # v4.5: position in batch
                    "input_tokens": in_tok,
                    "output_tokens":out_tok,
                })

                n_done += 1
                if fn_flag:                              n_fn   += 1
                if old_label == 1 and new_label == 0:   n_fp   += 1
                if not result["_parse_ok"]:              n_fail += 1

            # ── Flush checkpoint AFTER EVERY BATCH ───────────────────────────
            _total_now = time.time() - _pipeline_start_ts
            ckpt_ok, ckpt_saved_count, ckpt_last_idx = \
                _flush_checkpoint_with_meta(batch_buffer, _total_now, force=True)
            ckpt_flag    = ckpt_ok
            batch_buffer = []
            _batch_buffer_ref = batch_buffer  # keep emergency handler ref current

            # ── Flush OUTPUT after every batch (v4.5 NEW) ────────────────────
            _flush_output_file(new_rows)

            _la = pending_df["Tweet"].iloc[_row_cursor:_row_cursor+20].tolist()
            _a, _r, _t = _get_vram_stats()
            _next_bs   = _compute_dynamic_batch_size(_la, _r, _t)
            _upd(actual_bs_used, next_bs=_next_bs)

        # Trailing flush
        if batch_buffer:
            _total_now = time.time() - _pipeline_start_ts
            _, ckpt_saved_count, ckpt_last_idx = \
                _flush_checkpoint_with_meta(batch_buffer, _total_now, force=True)
            _flush_output_file(new_rows)
            batch_buffer = []
        _upd(actual_bs_used if "actual_bs_used" in dir() else BATCH_SIZE)
# --- Replace the end of Block E-C with this robust version ---
except KeyboardInterrupt:
    _stop = True
    logger.warning("Stop signal received. Finalizing current batch and saving...")
    if batch_buffer:
        _total_now = time.time() - _pipeline_start_ts
        _flush_checkpoint_with_meta(batch_buffer, _total_now, force=True)
        _flush_output_file(new_rows)
        batch_buffer = []

    # FORCE DRIVE SYNC (This prevents starting from an old checkpoint)
    print("Saving to Drive (flushing cache)...")
    os.system("sync")
    time.sleep(2)
    console.print("\n🛑 Interrupted. Progress is SAFELY saved to Drive. You can delete Block F.", style="bold green")

finally:
    # This runs if the PC shuts off or network crashes
    if 'batch_buffer' in locals() and batch_buffer:
        try:
            _total_now = time.time() - _pipeline_start_ts
            _flush_checkpoint_with_meta(batch_buffer, _total_now, force=True)
            _flush_output_file(new_rows)
            os.system("sync")
        except:
            pass
# ══════════════════════════════════════════════════════════════════════════════
# ASSEMBLE final_df
# ══════════════════════════════════════════════════════════════════════════════
_session_df = pd.DataFrame(new_rows) if new_rows else pd.DataFrame()
_all_df     = (pd.concat([done_df, _session_df], ignore_index=True)
               if len(done_df) > 0 else _session_df)
if "_orig_idx" in _all_df.columns:
    _all_df = (_all_df[_all_df["_orig_idx"] != -999]
               .drop_duplicates(subset=["_orig_idx"], keep="last")
               .sort_values("_orig_idx")
               .reset_index(drop=True))
final_df = _all_df.copy()

logger.info(
    f"final_df: {len(final_df):,} rows | "
    f"label dist: {final_df['label'].value_counts().to_dict()} | "
    f"fn rescued: {int(final_df['fn_flag'].sum())} | "
    f"parse_ok: {int(final_df['_parse_ok'].sum())}/{len(final_df)}"
)

_sess_el  = time.time() - session_start
_total_el = time.time() - _pipeline_start_ts
_eff_rate = (len(new_rows) / _sess_el * 60) if _sess_el > 5 else 0

def _fmt_t(s):
    m, se = divmod(int(s), 60)
    h, m  = divmod(m, 60)
    return f"{h}h {m}m {se}s" if h else f"{m}m {se}s"

console.rule("[bold bright_green]Block E-C v4.5 Complete[/bold bright_green]")
console.print(
    f"  • Session time          : [bright_cyan]{_fmt_t(_sess_el)}[/bright_cyan]\n"
    f"  • Total pipeline        : [white]{_fmt_t(_total_el)}[/white]\n"
    f"  • Tweets this session   : [white]{len(new_rows):,}[/white]\n"
    f"  • Effective rate        : [bright_cyan]{_eff_rate:.1f} tweets/min[/bright_cyan]\n"
    f"  • final_df rows         : [white]{len(final_df):,}[/white]\n"
    f"  • FN rescued            : [bright_green]{n_fn:,}[/bright_green]\n"
    f"  • FP flipped            : [yellow]{n_fp:,}[/yellow]\n"
    f"  • Parse errors          : {n_fail:,}\n"
    f"  • Rows on disk (ckpt)   : [bright_green]{ckpt_saved_count:,}[/bright_green]\n"
    f"  • OUTPUT file saved     : [bright_green]✅ after every batch[/bright_green]\n"
    f"  • F-A / F-B             : [dim]Optional — OUTPUT is already current[/dim]",
    style="yellow"
)
logger.info(
    f"✅ Block E-C v4.5 complete. "
    f"Session: {len(new_rows):,} tweets in {_fmt_t(_sess_el)}. "
    f"Rate: {_eff_rate:.1f} tweets/min. "
    f"OUTPUT updated after every batch."
)

====++++====
Block E-C v4.5 Output
====++++====

2026-04-19 11:08:16,559 | INFO | Block E-C v4.5 | pending=0 | COT=True | max_batch_size=12 | max_tokens/item=1024 | timeout=60s | ckpt_every=EVERY BATCH | output_every=EVERY BATCH | v4.5


Output()

──────────────────────────────────────────────────── Block E-C v4.5 Complete ─────────────────────────────────────────────────────

  • Session time          : 0m 0s
  • Total pipeline        : 23h 51m 13s
  • Tweets this session   : 0
  • Effective rate        : 0.0 tweets/min
  • final_df rows         : 5,095
  • FN rescued            : 684
  • FP flipped            : 996
  • Parse errors          : 0
  • Rows on disk (ckpt)   : 5,491
  • OUTPUT file saved     : ✅ after every batch
  • F-A / F-B             : Optional — OUTPUT is already current

In [ ]:
# ==========================
# Block F-A: GLM Pipeline — Rich Analysis Display
# Version: v2.0
# Purpose: Print all GLM pipeline analysis statistics (confusion matrix, category
#          distribution, confidence stats, FN/FP corrections) using a Rich UI identical
#          in quality to the finetuning pipeline's Block 11 display.
# ==========================
#
# What this block does: (~420 words)
#
# - Feature 1 (Rich Console Setup ✅): Instantiates a Rich Console at width=160,
#   matching the finetuning pipeline's UI width convention. All output goes through
#   this console — no bare print() calls anywhere in the analysis section. The console
#   uses the same style vocabulary (bright_green for positives, bright_yellow for warnings,
#   red for failures, cyan for VRAM/system stats, dim for secondary info) as Block 11.
#   This makes both pipelines' outputs visually consistent in the Colab cell.
#
# - Feature 2 (Confusion Matrix Table ✅): Renders the old_label vs judge_label 2×2
#   confusion matrix as a Rich Table with box.DOUBLE_EDGE framing. Cells are styled:
#   True positives (1→1) and true negatives (0→0) in bright_green; false positives
#   (0→1) and false negatives (1→0) in red. Column headers and row labels are bold.
#   This is the single most diagnostic output — too many red cells means the judge
#   prompt needs tuning.
#
# - Feature 3 (Category Distribution Table ✅): Renders value_counts of the category
#   column as a Rich Table with an inline percentage bar (█ characters, width=30).
#   Bars use bright_green for DEI categories (label=1 majority) and white for non-DEI.
#   The bar width is proportional to count / max_count so all bars fit in 160 chars.
#   This visualisation instantly shows which DEI subcategories are over/under-represented
#   in the final training set, guiding future data collection priorities.
#
# - Feature 4 (Confidence Statistics Panel ✅): Displays mean, median, and count of
#   rows with confidence < 50 inside a Rich Panel with title "Confidence Statistics".
#   If more than 5% of rows are low-confidence, the panel border turns red and a
#   warning message is logged explaining that the system prompt needs better examples.
#   This threshold mirrors the original Block F logic but is now visually prominent.
#
# - Feature 5 (FN/FP Correction Panel ✅): Shows FN corrections (old=0→new=1),
#   FP corrections (old=1→new=0), and agreed rows in a three-row Rich Panel. FN count
#   is styled bright_green (the primary deliverable of the pipeline). FP count triggers
#   a red warning if it exceeds 10% of original positives — same threshold as before,
#   now rendered as a styled Rich Text warning inside the panel rather than a bare logger
#   call, making it impossible to miss. Agreed count is styled dim white.
#
# - Feature 6 (Parse Failure Warning ✅): If _parse_ok=False rows exist, a distinct
#   red-bordered Rich Panel is rendered listing the count and percentage. Zero parse
#   failures renders a bright_green "✅ 0 parse failures" confirmation line. This
#   replaces the original bare logger.warning() call which was easy to scroll past.
#
# How it works:
# - Step 1: Import Rich components, get logger, define BASE_DIR and path constants.
# - Step 2: Validate final_df is in scope (raises RuntimeError if Block E did not run).
# - Step 3: Build and print confusion matrix Rich Table.
# - Step 4: Build and print category distribution Rich Table with bars.
# - Step 5: Compute and render confidence statistics in a Rich Panel.
# - Step 6: Compute and render FN/FP corrections in a Rich Panel.
# - Step 7: Render parse failure warning panel.
# - Step 8: Log all stats to training.log (file audit trail independent of Rich UI).
#
# Solutions attempted that did not work:
# - Failed attempt 1: Using Rich's built-in progress bar for the analysis section.
#   Analysis is instantaneous (pure pandas operations on 5k rows) so a progress bar
#   was misleading — it completed before the animation rendered. Replaced with
#   static Rich tables which render synchronously.
#
# Solutions implemented:
# - Solution 1: Static Rich Tables + Panels for all analysis. Progress bar reserved
#   for Block F-B where actual I/O latency exists (Drive writes take 2–15 seconds).

print("====++++====")
print("Block F-A Output")
print("====++++====")
print()

import os, logging
import pandas as pd
import numpy as np
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.text import Text
from rich.rule import Rule
from rich import box

logging.raiseExceptions = False
logger = logging.getLogger("dei_glm")
console = Console(width=160)

BASE_DIR    = "/content/drive/MyDrive/Twitter/Finetune"
OUTPUT      = os.path.join(BASE_DIR, "data/processed/glm_train_v3_4k.csv")
CHECKPOINT  = os.path.join(BASE_DIR, "data/judge/glm_train_v3_full_checkpoint.csv")
REVIEW_PATH = os.path.join(BASE_DIR, "data/processed/glm_train_v3_full_review.csv")
OUTPUT_COLS = ["Tweet","old_label","label","reasoning","category",
               "confidence","fn_flag","judge_agrees","source","_parse_ok"]

# ── Guard: final_df must be in scope from Block E ─────────────────────────────
if "final_df" not in dir() and "final_df" not in globals():
    raise RuntimeError(
        "final_df not found in session namespace. "
        "Block E v3.0 must be run before Block F-A."
    )
logger.info("Block F-A v2.0 — Rich Analysis Display starting.")
console.rule("[bold yellow]GLM Pipeline — Post-Classification Analysis[/bold yellow]")

# ── Feature 2: Confusion Matrix Table ────────────────────────────────────────
console.print()
console.rule("[bold white]OLD vs NEW LABEL COMPARISON MATRIX[/bold white]", style="grey50")

cross = pd.crosstab(
    final_df["old_label"], final_df["label"],
    rownames=["old_label"], colnames=["judge_label"]
)

cm_table = Table(box=box.DOUBLE_EDGE, show_header=True, header_style="bold magenta")
cm_table.add_column("old \\ judge", style="bold", justify="center")
for col in cross.columns:
    cm_table.add_column(f"Pred = {col}", justify="center")

for row_idx, row in cross.iterrows():
    cells = [f"True = {row_idx}"]
    for col_idx, val in row.items():
        # diagonal = agreement, off-diagonal = flip
        if row_idx == col_idx:
            cells.append(f"[bright_green]{val:,}[/bright_green]")
        else:
            cells.append(f"[red]{val:,}[/red]")
    cm_table.add_row(*cells)

console.print(cm_table)
logger.info(f"Confusion matrix:\n{cross.to_string()}")

# ── Feature 3: Category Distribution Table ────────────────────────────────────
console.print()
console.rule("[bold white]CATEGORY DISTRIBUTION (Judge Output)[/bold white]", style="grey50")

DEI_CATEGORIES = {
    "lgbtq", "race_ethnicity", "disability", "gender",
    "general_dei", "social_justice", "first_gen"
}

cat_counts = final_df["category"].value_counts()
max_count  = int(cat_counts.max()) if len(cat_counts) > 0 else 1
BAR_WIDTH  = 30

cat_table = Table(box=box.SIMPLE_HEAVY, show_header=True, header_style="bold cyan")
cat_table.add_column("Category",   style="bold",   min_width=20)
cat_table.add_column("Count",      justify="right", min_width=8)
cat_table.add_column("Pct",        justify="right", min_width=6)
cat_table.add_column("Distribution", min_width=BAR_WIDTH + 2)

for cat, cnt in cat_counts.items():
    pct      = cnt / len(final_df) * 100
    filled   = max(1, int(BAR_WIDTH * cnt / max_count))
    bar_char = "█" * filled + "░" * (BAR_WIDTH - filled)
    is_dei   = cat in DEI_CATEGORIES
    color    = "bright_green" if is_dei else "white"
    cat_table.add_row(
        f"[{color}]{cat}[/{color}]",
        f"[{color}]{cnt:,}[/{color}]",
        f"[{color}]{pct:.1f}%[/{color}]",
        f"[{color}]{bar_char}[/{color}]",
    )

console.print(cat_table)
logger.info(f"Category distribution: {cat_counts.to_dict()}")

# ── Feature 4: Confidence Statistics Panel ────────────────────────────────────
console.print()
mean_conf   = final_df["confidence"].mean()
median_conf = final_df["confidence"].median()
n_low_conf  = (final_df["confidence"] < 50).sum()
low_pct     = 100 * n_low_conf / max(len(final_df), 1)
border_color = "red" if low_pct > 5.0 else "bright_green"

conf_text = Text()
conf_text.append(f"  Mean confidence   : ", style="dim white")
conf_text.append(f"{mean_conf:.1f}\n", style="bold white")
conf_text.append(f"  Median confidence : ", style="dim white")
conf_text.append(f"{median_conf:.1f}\n", style="bold white")
conf_text.append(f"  Rows conf < 50    : ", style="dim white")
low_style = "bold red" if low_pct > 5.0 else "bold bright_green"
conf_text.append(f"{n_low_conf:,}  ({low_pct:.1f}%)", style=low_style)
if low_pct > 5.0:
    conf_text.append(
        f"\n\n  ⚠  Low-confidence rate exceeds 5% threshold.\n"
        f"     Add more examples to JUDGE_SYSTEM_PROMPT for ambiguous categories.",
        style="yellow"
    )
    logger.warning(
        f"Low-confidence rate {low_pct:.1f}% > 5% threshold. "
        "Consider adding more examples to JUDGE_SYSTEM_PROMPT for ambiguous categories."
    )

console.print(Panel(conf_text, title="[bold]Confidence Statistics[/bold]",
                    border_style=border_color))
logger.info(f"Confidence — mean: {mean_conf:.1f} | median: {median_conf:.1f} | "
            f"low(<50): {n_low_conf} ({low_pct:.1f}%)")

# ── Feature 5: FN/FP Correction Panel ────────────────────────────────────────
console.print()
n_fn_corrections = int(final_df["fn_flag"].sum())
n_fp_corrections = int(
    ((final_df["old_label"] == 1) & (final_df["label"] == 0)).sum()
)
n_agreed  = int((final_df["old_label"] == final_df["label"]).sum())
n_pos_orig = int((final_df["old_label"] == 1).sum())
fp_rate    = n_fp_corrections / max(n_pos_orig, 1)

correction_text = Text()
correction_text.append("  FN corrections  (old=0 → new=1, fn_flag=1) : ", style="dim white")
correction_text.append(f"{n_fn_corrections:,}\n", style="bold bright_green")
correction_text.append("  FP corrections  (old=1 → new=0)            : ", style="dim white")
fp_style = "bold red" if fp_rate > 0.10 else "bold white"
correction_text.append(f"{n_fp_corrections:,}", style=fp_style)
if fp_rate > 0.10:
    correction_text.append(
        f"  ← {fp_rate*100:.1f}% of original positives flipped!\n"
        f"     Judge may be more conservative than original labeler. Review FP rows manually.",
        style="yellow"
    )
    logger.warning(
        f"FP correction rate {fp_rate*100:.1f}% > 10%. "
        "Judge may be more conservative than original labeler. Review FP rows manually."
    )
else:
    correction_text.append("\n", style="white")
correction_text.append("  Agreed          (old == new)                : ", style="dim white")
correction_text.append(f"{n_agreed:,}", style="dim white")

fp_border = "red" if fp_rate > 0.10 else "bright_green"
console.print(Panel(correction_text, title="[bold]Label Correction Summary[/bold]",
                    border_style=fp_border))
logger.info(f"FN corrections: {n_fn_corrections} | FP corrections: {n_fp_corrections} | "
            f"Agreed: {n_agreed}")

# ── Feature 6: Parse Failure Panel ────────────────────────────────────────────
console.print()
n_parse_fail = int((~final_df["_parse_ok"]).sum())
fail_pct     = 100 * n_parse_fail / max(len(final_df), 1)

if n_parse_fail == 0:
    parse_text = Text("  ✅  0 parse failures — all rows successfully classified.", style="bold bright_green")
    parse_border = "bright_green"
else:
    parse_text = Text()
    parse_text.append(f"  ❌  {n_parse_fail:,} parse failures ({fail_pct:.1f}%)\n", style="bold red")
    parse_text.append(
        "     These rows have label=old_label (fallback). "
        "Inspect REVIEW file before using in training.",
        style="yellow"
    )
    parse_border = "red"
    logger.warning(f"Parse failures: {n_parse_fail} ({fail_pct:.1f}%)")

console.print(Panel(parse_text, title="[bold]Parse Failure Report[/bold]",
                    border_style=parse_border))

logger.info("✅ Block F-A complete. Proceed to Block F-B for file save.")
console.print()
console.rule("[bold bright_green]Block F-A Complete — Run Block F-B to save files[/bold bright_green]")

In [ ]:
# ==========================
# Block F-B: GLM Pipeline — Atomic Save, Review Export, Summary Panel
# Version: v2.0
# Purpose: Atomically save glm_train_v3_full.csv, export the review file for low-confidence
#          and parse-failure rows, delete the checkpoint, and render the final Rich summary
#          panel identical in style to Block 11d's session summary.
# ==========================
#
# What this block does: (~400 words)
#
# - Feature 1 (Rich Progress for File Saves ✅): Wraps both atomic file-write operations
#   (OUTPUT and REVIEW_PATH) inside a single Rich Progress context with SpinnerColumn,
#   TextColumn showing the target filename, BarColumn, and TaskProgressColumn. Each save
#   is one task (0 → 1 step). The spinner animates during the Drive write (2–15 seconds
#   on a slow mount) so the user sees live feedback instead of a frozen cell. This is
#   the same pattern used by Block 11d for per-file inference progress, applied here to
#   I/O operations rather than model inference.
#
# - Feature 2 (Atomic OUTPUT Save ✅): Writes final_df[OUTPUT_COLS] to OUTPUT + ".tmp"
#   then calls os.replace() to atomically swap it to OUTPUT. The .tmp file is cleaned up
#   in a finally block regardless of success or failure. If os.replace() fails (Drive
#   FUSE mount error), the .tmp is preserved and the error is raised so the checkpoint
#   is NOT deleted. This is the same .tmp → os.replace() pattern from Block F v1.0 but
#   now wrapped in a Rich Progress task so Drive latency is visible.
#
# - Feature 3 (Checkpoint Deletion ✅): Deletes the checkpoint file ONLY after a
#   confirmed successful os.replace() call for the OUTPUT file. If OUTPUT save fails,
#   the checkpoint is preserved so the pipeline can be resumed. The deletion is logged
#   to training.log with the checkpoint path for audit trail.
#
# - Feature 4 (Review File Export ✅): Builds a review DataFrame of rows where
#   _parse_ok=False OR confidence < 50. These are the rows with fallback labels or
#   low judge confidence that need manual inspection before use in training. Saved to
#   REVIEW_PATH with the same .tmp → os.replace() atomic pattern. If the review
#   DataFrame is empty (perfect run), logs a bright_green "✅ No review rows needed"
#   message and skips the write.
#
# - Feature 5 (Final Summary Panel ✅): Renders a Rich Panel styled identically to
#   Block 11d's "SESSION SUMMARY" panel. Includes: total rows, label distribution,
#   FN corrections, FP corrections, parse failure rate, low-confidence rate, output
#   path, and review path. Panel border is bright_green on clean runs, yellow if
#   warnings were triggered (parse failures > 2%, low confidence > 5%, FP rate > 10%).
#   This panel is the last thing visible in the cell — it acts as the go/no-go signal
#   before using the output file in training.
#
# - Feature 6 (Dual Log ✅): All key metrics are also written to training.log via the
#   dei_glm logger (FileHandler). The Rich panel is transient in Colab cells — the log
#   file is the permanent audit record. Both records contain identical information.
#
# How it works:
# - Step 1: Import Rich Progress components and path constants.
# - Step 2: Open Progress context, add OUTPUT save task, execute .tmp → os.replace().
# - Step 3: Delete checkpoint inside the same Progress context on success.
# - Step 4: Add REVIEW save task, build review_df, execute save if non-empty.
# - Step 5: Close Progress context (spinner stops automatically).
# - Step 6: Compute summary stats and render final Rich Panel.
# - Step 7: Write all stats to training.log.
#
# Solutions attempted that did not work:
# - Failed attempt 1: Using rich.progress.track() on a list of two file paths.
#   track() requires an iterable and renders a single bar for the whole iterable,
#   not per-task. Two separate Progress tasks give cleaner per-file feedback.
# - Failed attempt 2: Deleting checkpoint inside a finally block. If OUTPUT save
#   raises, finally runs the deletion anyway, destroying the resume point.
#   Fixed by moving deletion to the success branch only (after os.replace()).
#
# Solutions implemented:
# - Solution 1: Rich Progress with two explicit tasks (OUTPUT, REVIEW). Checkpoint
#   deletion gated on confirmed OUTPUT save success.
# - Solution 2: Panel border color computed from warning flags so the summary
#   panel is self-explanatory without reading the log.

print("====++++====")
print("Block F-B Output")
print("====++++====")
print()

import os, logging, time
import pandas as pd
import numpy as np
from rich.console import Console
from rich.panel import Panel
from rich.text import Text
from rich.rule import Rule
from rich.progress import (
    Progress, SpinnerColumn, TextColumn,
    BarColumn, TaskProgressColumn, TimeElapsedColumn,
)
from rich import box

logging.raiseExceptions = False
logger = logging.getLogger("dei_glm")
console = Console(width=160)

BASE_DIR    = "/content/drive/MyDrive/Twitter/Finetune"
OUTPUT      = os.path.join(BASE_DIR, "data/processed/glm_train_v3_4k.csv")
CHECKPOINT  = os.path.join(BASE_DIR, "data/judge/glm_train_v3_full_checkpoint.csv")
REVIEW_PATH = os.path.join(BASE_DIR, "data/processed/glm_train_v3_full_review.csv")
OUTPUT_COLS = ["Tweet","old_label","label","reasoning","category",
               "confidence","fn_flag","judge_agrees","source","_parse_ok"]

# ── Guard: final_df must be in scope from Block E ─────────────────────────────
if "final_df" not in dir() and "final_df" not in globals():
    raise RuntimeError(
        "final_df not found in session namespace. "
        "Block E v3.0 must be run before Block F-B."
    )

# ── Re-derive stats needed for summary panel ──────────────────────────────────
# These are recomputed here so Block F-B can run independently if F-A was skipped.
n_fn_corrections = int(final_df["fn_flag"].sum())
n_fp_corrections = int(
    ((final_df["old_label"] == 1) & (final_df["label"] == 0)).sum()
)
n_parse_fail  = int((~final_df["_parse_ok"]).sum())
fail_pct      = 100 * n_parse_fail / max(len(final_df), 1)
n_low_conf    = int((final_df["confidence"] < 50).sum())
low_pct       = 100 * n_low_conf / max(len(final_df), 1)
n_pos_orig    = int((final_df["old_label"] == 1).sum())
fp_rate       = n_fp_corrections / max(n_pos_orig, 1)
label_dist    = final_df["label"].value_counts().to_dict()

# Warning flag: any threshold exceeded → yellow border on summary panel
_has_warning  = (fail_pct > 2.0) or (low_pct > 5.0) or (fp_rate > 0.10)

# ── Review DataFrame ──────────────────────────────────────────────────────────
review_mask = (~final_df["_parse_ok"]) | (final_df["confidence"] < 50)
review_df   = final_df[review_mask].copy()

# ── Feature 1–4: Rich Progress for all I/O ───────────────────────────────────
console.rule("[bold white]Saving Files to Google Drive[/bold white]", style="grey50")
console.print()

output_saved = False
review_saved = False

with Progress(
    SpinnerColumn(),
    TextColumn("[bold cyan]{task.description}"),
    BarColumn(bar_width=40),
    TaskProgressColumn(),
    TimeElapsedColumn(),
    console=console,
    transient=False,
) as progress:

    # ── Task 1: Save OUTPUT ───────────────────────────────────────────────────
    t_output = progress.add_task(
        f"Saving OUTPUT  → {os.path.basename(OUTPUT)}", total=1
    )
    _tmp_out = OUTPUT + ".tmp"
    try:
        final_df[OUTPUT_COLS].to_csv(_tmp_out, index=False, encoding="utf-8")
        os.replace(_tmp_out, OUTPUT)
        output_saved = True
        logger.info(f"✅ Saved {len(final_df):,} rows to: {OUTPUT}")
        progress.advance(t_output)
        # ── Checkpoint deletion (only on confirmed save) ──────────────────────
        if os.path.isfile(CHECKPOINT):
            os.remove(CHECKPOINT)
            logger.info(f"Checkpoint deleted: {CHECKPOINT}")
    except Exception as _save_err:
        logger.error(f"OUTPUT SAVE FAILED: {_save_err}. Checkpoint preserved at {CHECKPOINT}.")
        raise
    finally:
        if os.path.exists(_tmp_out):
            try: os.remove(_tmp_out)
            except: pass

    # ── Task 2: Save REVIEW ───────────────────────────────────────────────────
    t_review = progress.add_task(
        f"Saving REVIEW  → {os.path.basename(REVIEW_PATH)}", total=1
    )
    if len(review_df) == 0:
        logger.info("✅ No review rows — all rows parsed OK with confidence ≥ 50.")
        review_saved = True
        progress.advance(t_review)
    else:
        _tmp_rev = REVIEW_PATH + ".tmp"
        try:
            review_df.to_csv(_tmp_rev, index=False, encoding="utf-8")
            os.replace(_tmp_rev, REVIEW_PATH)
            review_saved = True
            logger.info(f"Review file saved: {len(review_df):,} rows → {REVIEW_PATH}")
            progress.advance(t_review)
        except Exception as _rev_err:
            logger.warning(f"Review file save failed: {_rev_err}")
        finally:
            if os.path.exists(_tmp_rev):
                try: os.remove(_tmp_rev)
                except: pass

console.print()

# ── Feature 5: Final Summary Panel ───────────────────────────────────────────
summary_color = "yellow" if _has_warning else "bright_green"
summary_icon  = "⚠️ " if _has_warning else "🏁"

summary_text = Text()
summary_text.append(f"  {summary_icon}  GLM Pipeline Run Complete\n\n", style="bold white")

summary_text.append("  Total rows in output    : ", style="dim white")
summary_text.append(f"{len(final_df):,}\n", style="bold white")

summary_text.append("  Label distribution      : ", style="dim white")
summary_text.append(
    f"{{0: {label_dist.get(0, 0):,}, 1: {label_dist.get(1, 0):,}}}\n",
    style="bold white"
)

summary_text.append("  FN corrections          : ", style="dim white")
summary_text.append(f"{n_fn_corrections:,}", style="bold bright_green")
summary_text.append("  (old=0 → new=1)\n", style="dim")

summary_text.append("  FP corrections          : ", style="dim white")
fp_col = "bold red" if fp_rate > 0.10 else "bold white"
summary_text.append(f"{n_fp_corrections:,}\n", style=fp_col)

summary_text.append("  Parse failures          : ", style="dim white")
pf_col = "bold red" if fail_pct > 2.0 else "bold bright_green"
summary_text.append(f"{n_parse_fail:,}  ({fail_pct:.1f}%)\n", style=pf_col)

summary_text.append("  Low confidence (<50)    : ", style="dim white")
lc_col = "bold yellow" if low_pct > 5.0 else "bold white"
summary_text.append(f"{n_low_conf:,}  ({low_pct:.1f}%)\n", style=lc_col)

summary_text.append("\n  Output file             : ", style="dim white")
out_col = "bright_green" if output_saved else "red"
summary_text.append(f"{'✅' if output_saved else '❌'}  {OUTPUT}\n", style=out_col)

summary_text.append("  Review file             : ", style="dim white")
rev_display = (
    f"✅  {REVIEW_PATH}  ({len(review_df):,} rows)" if (review_saved and len(review_df) > 0)
    else ("✅  No review file needed (0 rows)" if (review_saved and len(review_df) == 0)
    else f"❌  SAVE FAILED — {REVIEW_PATH}")
)
rev_col = "bright_green" if review_saved else "red"
summary_text.append(f"{rev_display}", style=rev_col)

console.print(Panel(
    summary_text,
    title=f"[bold {summary_color}]GLM Pipeline Final Summary[/bold {summary_color}]",
    border_style=summary_color,
    padding=(1, 2),
))

# ── Feature 6: Dual Log ───────────────────────────────────────────────────────
logger.info("=" * 60)
logger.info("GLM PIPELINE FINAL SUMMARY")
logger.info("=" * 60)
logger.info(f"Total rows in output    : {len(final_df):,}")
logger.info(f"Label dist (new)        : {label_dist}")
logger.info(f"FN corrections          : {n_fn_corrections}")
logger.info(f"FP corrections          : {n_fp_corrections}")
logger.info(f"Parse failures          : {n_parse_fail} ({fail_pct:.1f}%)")
logger.info(f"Low confidence (<50)    : {n_low_conf} ({low_pct:.1f}%)")
logger.info(f"Output file             : {OUTPUT}")
logger.info(f"Review file             : {REVIEW_PATH}")
logger.info("✅ Block F-B complete. GLM pipeline finished.")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# 1. SETUP CONFIGURATION (ADD NEW MODELS HERE)
# ==========================================
output_plot_dir = '/content/drive/MyDrive/Twitter/Finetune/inference/plots'
os.makedirs(output_plot_dir, exist_ok=True)

# Define your models.
# The script uses the first model in this list ("Qwen 3.5 4B") as the master list.
MODELS_CONFIG = {
    "Qwen 3.5 4B": {
        "folder": "/content/drive/MyDrive/Twitter/Finetune/inference/output/qwen3.5_4b",
        "file_prefix": "classified_",
        "file_suffix": ".csv",
        "pred_col": "dei_label",
        "color": "orange",
        "marker": "s"
    },
    "Gemini 3 Flash Preview": {
        "folder": "/content/drive/MyDrive/Twitter/New_everything/Colab/Files/gemini-3-flash-preview/merge",
        "file_prefix": "",
        "file_suffix": ".csv",
        "pred_col": "gemini_label",
        "color": "green",
        "marker": "^"
    },
    "Qwen 3.5 4B Merged": {
        "folder": "/content/drive/MyDrive/Twitter/Finetune/inference/output/qwen3.5_4b_merged",
        "file_prefix": "classified_",
        "file_suffix": ".csv",
        "pred_col": "dei_label", # Change if the merged file uses a different column name
        "color": "blue",
        "marker": "o"
    }
}

# ==========================================
# 2. GET EXACT LIST OF BASE UNIVERSITIES
# ==========================================
global_yearly_data = {model: pd.DataFrame() for model in MODELS_CONFIG.keys()}
global_totals = {model: {"total_rows": 0, "total_pos": 0} for model in MODELS_CONFIG.keys()}

# WE SPECIFICALLY USE QWEN AS THE MASTER LIST (The ~14-16 files)
base_model_key = "Qwen 3.5 4B"
base_folder = MODELS_CONFIG[base_model_key]["folder"]
base_prefix = MODELS_CONFIG[base_model_key]["file_prefix"]

uni_names = []
if os.path.exists(base_folder):
    for f in os.listdir(base_folder):
        if f.endswith('.csv'):
            # Strip prefix and suffix to get pure university name
            uni_name = f.replace(base_prefix, "").replace('.csv', "")
            uni_names.append(uni_name)
else:
    print(f"ERROR: Base folder not found: {base_folder}")

print(f"Found {len(uni_names)} universities in the base directory.")

# ==========================================
# 3. PROCESS AND PLOT EACH UNIVERSITY
# ==========================================
for uni in uni_names:
    plt.figure(figsize=(10, 6))

    all_years = set()       # Track all years present for this university
    model_uni_data = {}     # Store data to plot later

    # Read data for all models for this specific university
    for model_name, cfg in MODELS_CONFIG.items():
        filename = f"{cfg['file_prefix']}{uni}{cfg['file_suffix']}"
        filepath = os.path.join(cfg['folder'], filename)

        if not os.path.exists(filepath):
            print(f"  -> Warning: Missing {model_name} file for {uni}. Skipping this line on the chart.")
            continue

        df = pd.read_csv(filepath)
        total_rows = len(df)

        # Ensure 'Date' is datetime and extract 'Year'
        df['Year'] = pd.to_datetime(df['Date'], errors='coerce').dt.year
        # Drop rows where year is missing (NaT)
        df = df.dropna(subset=['Year'])
        df['Year'] = df['Year'].astype(int)

        # Aggregate Positive Counts
        agg = df.groupby('Year')[cfg['pred_col']].sum().reset_index(name='Count')
        total_pos = agg['Count'].sum()

        # Track globally for the combined final plot
        global_totals[model_name]["total_rows"] += total_rows
        global_totals[model_name]["total_pos"] += total_pos
        global_yearly_data[model_name] = pd.concat([global_yearly_data[model_name], agg])

        # Save for local plotting
        model_uni_data[model_name] = {
            "agg": agg,
            "total_rows": total_rows,
            "total_pos": total_pos,
            "cfg": cfg
        }
        all_years.update(agg['Year'].unique())

    # If no data was loaded at all for this uni, skip plotting
    if not model_uni_data:
        plt.close()
        continue

    sorted_years = sorted(list(all_years))

    # Plot lines for this university
    for model_name, data in model_uni_data.items():
        agg = data["agg"]
        cfg = data["cfg"]

        # Reindex to ensure missing years are filled with 0 (aligns the chart properly)
        agg = agg.set_index('Year').reindex(sorted_years, fill_value=0).reset_index()

        # Calculate Legend Label
        pct = (data["total_pos"] / data["total_rows"] * 100) if data["total_rows"] > 0 else 0
        label = f'{model_name} ({int(data["total_pos"])}/{data["total_rows"]} ({pct:.1f}%))'

        plt.plot(agg['Year'], agg['Count'], marker=cfg['marker'], label=label, color=cfg['color'])

    # Format the Individual Plot
    clean_uni_name = uni.replace('_', ' ').title()
    plt.title(f"{clean_uni_name}", fontsize=14, fontweight='bold')
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("Positive DEI Count", fontsize=12)
    plt.xticks(sorted_years)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()

    # Save Individual Plot
    save_path = os.path.join(output_plot_dir, f"{uni}_plot.png")
    plt.savefig(save_path)
    plt.close()
    print(f"Saved plot for: {uni}")

# ==========================================
# 4. PLOT COMBINED DATA (ALL UNIVERSITIES)
# ==========================================
print("\nGenerating Combined Final Plot...")
plt.figure(figsize=(12, 7))

global_all_years = set()
global_processed_data = {}

# Aggregate Global Data
for model_name, cfg in MODELS_CONFIG.items():
    df_all = global_yearly_data[model_name]

    if df_all.empty:
        continue

    # Group the stacked dataset by Year globally
    df_grouped = df_all.groupby('Year')['Count'].sum().reset_index()
    global_all_years.update(df_grouped['Year'].unique())

    global_processed_data[model_name] = {
        "agg": df_grouped,
        "cfg": cfg
    }

# Plot Global Data
if global_processed_data:
    sorted_global_years = sorted(list(global_all_years))

    for model_name, data in global_processed_data.items():
        agg = data["agg"]
        cfg = data["cfg"]

        # Reindex for alignment
        agg = agg.set_index('Year').reindex(sorted_global_years, fill_value=0).reset_index()

        # Calculate Global Label
        t_rows = global_totals[model_name]["total_rows"]
        t_pos = global_totals[model_name]["total_pos"]
        pct = (t_pos / t_rows * 100) if t_rows > 0 else 0

        g_label = f'{model_name} ({int(t_pos)}/{t_rows} ({pct:.1f}%))'

        plt.plot(agg['Year'], agg['Count'], marker=cfg['marker'], linewidth=2.5, label=g_label, color=cfg['color'])

    # Format Aggregate Plot
    plt.title(f"Total Positive DEI Count Over Time ({len(uni_names)} Universities)", fontsize=16, fontweight='bold')
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("Total Positive DEI Count", fontsize=12)
    plt.xticks(sorted_global_years)
    plt.legend(fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()

    # Save Aggregate Plot
    agg_save_path = os.path.join(output_plot_dir, "Aggregate_Combined_Plot.png")
    plt.savefig(agg_save_path)
    plt.show()

print(f"\nAll plots successfully saved in: {output_plot_dir}")

# Disagreement analysis

In [ ]:
import os
import pandas as pd
import random

# ==========================================
# 1. SETUP CONFIGURATION
# ==========================================
output_excel_path = '/content/drive/MyDrive/Twitter/Finetune/inference/plots/Model_Disagreement_Analysis.xlsx'

# Define folder paths and column names
MODELS_CONFIG = {
    "Qwen_Base": {
        "folder": "/content/drive/MyDrive/Twitter/Finetune/inference/output/qwen3.5_4b",
        "file_prefix": "classified_",
        "file_suffix": ".csv",
        "pred_col": "dei_label"
    },
    "Qwen_Merged": {
        "folder": "/content/drive/MyDrive/Twitter/Finetune/inference/output/qwen3.5_4b_merged",
        "file_prefix": "classified_",
        "file_suffix": ".csv",
        "pred_col": "dei_label"
    },
    "Gemini": {
        "folder": "/content/drive/MyDrive/Twitter/New_everything/Colab/Files/gemini-3-flash-preview/merge",
        "file_prefix": "",
        "file_suffix": ".csv",
        "pred_col": "gemini_label"
    }
}

base_folder = MODELS_CONFIG["Qwen_Base"]["folder"]
base_prefix = MODELS_CONFIG["Qwen_Base"]["file_prefix"]

# ==========================================
# 2. COLLECT DISAGREEMENTS
# ==========================================
all_qwen0_gem1 = pd.DataFrame()
all_qwen1_gem0 = pd.DataFrame()

# Get base university names from filenames
uni_names = []
for f in os.listdir(base_folder):
    if f.endswith('.csv'):
        uni_names.append(f.replace(base_prefix, "").replace('.csv', ""))

print(f"Scanning {len(uni_names)} files for disagreements...\n")

for uni in uni_names:
    path_qb = os.path.join(MODELS_CONFIG["Qwen_Base"]["folder"], f"{MODELS_CONFIG['Qwen_Base']['file_prefix']}{uni}{MODELS_CONFIG['Qwen_Base']['file_suffix']}")
    path_qm = os.path.join(MODELS_CONFIG["Qwen_Merged"]["folder"], f"{MODELS_CONFIG['Qwen_Merged']['file_prefix']}{uni}{MODELS_CONFIG['Qwen_Merged']['file_suffix']}")
    path_gem = os.path.join(MODELS_CONFIG["Gemini"]["folder"], f"{MODELS_CONFIG['Gemini']['file_prefix']}{uni}{MODELS_CONFIG['Gemini']['file_suffix']}")

    if not (os.path.exists(path_qb) and os.path.exists(path_qm) and os.path.exists(path_gem)):
        print(f"  -> Skipping {uni}: Missing one or more model files.")
        continue

    df_qb = pd.read_csv(path_qb)
    df_qm = pd.read_csv(path_qm)
    df_gem = pd.read_csv(path_gem)

    if not (len(df_qb) == len(df_qm) == len(df_gem)):
        print(f"  -> Warning: Row counts don't match for {uni}. Skipping to avoid misaligned text.")
        continue

    # --- BULLETPROOF TEXT COLUMN FINDER ---
    possible_text_cols = ['Tweet', 'tweet', 'Text', 'text', 'full_text', 'content']
    text_col = None

    for col in possible_text_cols:
        if col in df_qb.columns:
            text_col = col
            break

    # If it still can't find it, pick the first column that IS NOT 'University', 'Date', or a Label
    if text_col is None:
        ignore_cols = ['University', 'Date', 'Unnamed: 0', 'dei_label', 'gemini_label', 'ID', 'id']
        for col in df_qb.columns:
            if col not in ignore_cols:
                text_col = col
                break

    # Extract University name (Prioritize the actual CSV column if it exists, otherwise use filename)
    uni_column_data = df_qb['University'] if 'University' in df_qb.columns else uni

    # Create combined dataframe
    combined_df = pd.DataFrame({
        'University': uni_column_data,
        'Date': df_qb['Date'] if 'Date' in df_qb.columns else 'Unknown',
        'Tweet_Text': df_qb[text_col],
        'Qwen_Base_Label': df_qb[MODELS_CONFIG['Qwen_Base']['pred_col']],
        'Qwen_Merged_Label': df_qm[MODELS_CONFIG['Qwen_Merged']['pred_col']],
        'Gemini_Label': df_gem[MODELS_CONFIG['Gemini']['pred_col']]
    })

    # Condition 1: All Qwen = 0, Gemini = 1
    cond1 = (combined_df['Qwen_Base_Label'] == 0) & (combined_df['Qwen_Merged_Label'] == 0) & (combined_df['Gemini_Label'] == 1)
    all_qwen0_gem1 = pd.concat([all_qwen0_gem1, combined_df[cond1]])

    # Condition 2: All Qwen = 1, Gemini = 0
    cond2 = (combined_df['Qwen_Base_Label'] == 1) & (combined_df['Qwen_Merged_Label'] == 1) & (combined_df['Gemini_Label'] == 0)
    all_qwen1_gem0 = pd.concat([all_qwen1_gem0, combined_df[cond2]])

print(f"Total disagreements found:")
print(f"  - Qwen=0, Gemini=1: {len(all_qwen0_gem1)} tweets")
print(f"  - Qwen=1, Gemini=0: {len(all_qwen1_gem0)} tweets")

# ==========================================
# 3. SAMPLE AND EXPORT TO EXCEL
# ==========================================
sample_size = 100

final_qwen0_gem1 = all_qwen0_gem1.sample(n=min(sample_size, len(all_qwen0_gem1)), random_state=42)
final_qwen1_gem0 = all_qwen1_gem0.sample(n=min(sample_size, len(all_qwen1_gem0)), random_state=42)

print("\nGenerating Excel File...")

with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
    final_qwen0_gem1.to_excel(writer, sheet_name='Qwen_0_Gemini_1', index=False)
    final_qwen1_gem0.to_excel(writer, sheet_name='Qwen_1_Gemini_0', index=False)

print(f"\nSuccess! Disagreement Analysis dataset saved to:\n{output_excel_path}")